# NPO local third run — complete LoRA experiment

**This one notebook contains the entire experiment.** Copy this `.ipynb` to your
Windows PC; no repository Python files, `config.json` or input registry are
needed. Model weights and the six original TOFU data files remain external.
All training, tests, evaluation, quantization, reporting and optional run-2
reanalysis code is visible in the cells below. Nothing writes or launches a
Python script. Logs appear directly beneath the running Jupyter cell.

Use the same CUDA-enabled Python environment as run 2. Edit **Your settings**,
save the notebook, restart the kernel and choose **Run All**. Definitions and
engineering tests run first, then the actual three-seed experiment. Run All
starts real work: allow time for six controls and periodic full evaluations.
Keep at least 12 GiB free on the output SSD. The actual 1B Windows GPU runtime
and memory requirements still need measurement.

This is a declared **LoRA adaptation of NPO+RT**, not the paper's full-weight
reproduction. The NPO objective, positive retain CE, beta, coefficients and
source-rounded schedule remain unchanged. LoRA and fresh retain sampling are
explicit project choices (DL-024). The original base is still recoverable if
the unmerged adapter is disabled. We evaluate the adapter, merge in FP32,
save/reload BF16 and independently quantize those merged weights.

Primary outputs: geometric TOFU Forget Quality against fixed and matched
retain95 oracles; arithmetic nine-component Model Utility; gold-answer
likelihoods; ROUGE/F1/EM; paired directional CIs; KL; actual completions;
EOS/cap rates; merge drift and cross-seed tables. Greedy96 and long256 are
separate generation profiles. MIA/FADE/relearning and semantic human labels
remain explicitly unmeasured until their own studies are performed.

Formula: [NPO Eq. 3](https://arxiv.org/html/2404.05868v2).
Metric distinction: [TOFU utils](https://github.com/locuslab/tofu/blob/main/utils.py).

## 1. Environment

Select your working FYP CUDA kernel. This notebook does **not** reinstall
PyTorch or CUDA. In that environment, add any missing packages before running:

```powershell
python -m pip install peft accelerate huggingface-hub safetensors numpy pandas scipy matplotlib rouge-score psutil pytest
```

INT8/NF4 also need a compatible CUDA-enabled bitsandbytes installation.
Intended ranges: torch 2.6–2.x, transformers 4.51.3–5.x, peft 0.15.2–0.x,
accelerate 1.6–1.x and bitsandbytes 0.46.1–0.x. These are compatibility intent,
not a universal lock. Keep the run-2 working CUDA stack and run the tests below.

In [ ]:
import importlib.util
import sys

# This checks the selected kernel; it never installs or replaces PyTorch/CUDA.
DEPENDENCIES = {
    "torch": "torch",
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "huggingface_hub": "huggingface-hub",
    "safetensors": "safetensors",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "rouge_score": "rouge-score",
    "psutil": "psutil",
    "pytest": "pytest",
    "IPython": "ipython",
}
missing = [
    package for module, package in DEPENDENCIES.items() if importlib.util.find_spec(module) is None
]
print("Selected kernel:", sys.executable)
if missing:
    print("Missing packages:", ", ".join(missing))
    print("Install into this same kernel, then restart and Run All:")
    print("%pip install " + " ".join(missing))
    raise ModuleNotFoundError("Missing notebook dependencies: " + ", ".join(missing))
print(
    "Notebook dependencies found. INT8/NF4 additionally require a working bitsandbytes CUDA backend."
)

In [ ]:
import ast
import contextlib
import copy
import gc
import hashlib
import html
import importlib.metadata
import inspect
import io
import json
import math
import os
import platform
import random
import re
import shutil
import sys
import time
import traceback
import unicodedata
from collections import Counter
from contextlib import contextmanager, redirect_stderr, redirect_stdout
from dataclasses import asdict, dataclass, field
from enum import Enum
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import pytest
import torch
import torch.nn.functional as functional
from IPython.display import HTML, display
from peft import LoraConfig, TaskType, get_peft_model, set_peft_model_state_dict
from rouge_score import rouge_scorer
from scipy.special import logsumexp
from scipy.stats import ks_2samp
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    LlamaConfig,
    LlamaForCausalLM,
    PreTrainedTokenizerFast,
)

# Usually leave this empty. Set it only if your kernel starts outside the
# notebook directory/repository root, or if you rename the notebook.
NOTEBOOK_FILE = None  # e.g. r"C:\FYP\NPO_3RD_RUN.ipynb"
candidates = (
    [Path(NOTEBOOK_FILE)]
    if NOTEBOOK_FILE
    else [
        Path.cwd() / "NPO_3RD_RUN.ipynb",
        Path.cwd() / "experiments/reproductions/NPO_3rd-test/NPO_3RD_RUN.ipynb",
    ]
)
NOTEBOOK_PATH = next((p.resolve() for p in candidates if p.is_file()), None)
if NOTEBOOK_PATH is None:
    raise FileNotFoundError(
        "Set NOTEBOOK_FILE to this saved .ipynb file, then restart and Run All."
    )
NOTEBOOK_HOME = NOTEBOOK_PATH.parent


def read_saved_core_sources():
    document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    return {
        cell["id"]: "".join(cell["source"])
        for cell in document["cells"]
        if cell["cell_type"] == "code" and "npo3-core" in cell.get("metadata", {}).get("tags", [])
    }


SAVED_CORE_SOURCES = read_saved_core_sources()
NOTEBOOK_CODE = {
    name: hashlib.sha256(source.encode()).hexdigest() for name, source in SAVED_CORE_SOURCES.items()
}
REGISTERED_CORE_CELLS = set()
_NOTEBOOK_OBSERVER = None


def register_notebook_cell(identifier):
    # Jupyter records input before executing it. Reject unsaved scientific
    # cell edits rather than attributing them to different bytes on disk.
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    current = (
        shell.history_manager.input_hist_raw[-1]
        if shell is not None
        else globals().get("_VALIDATING_CELL_SOURCE")
    )
    if current is not None:
        expected = ast.dump(ast.parse(SAVED_CORE_SOURCES[identifier]))
        if ast.dump(ast.parse(current)) != expected:
            raise ValueError(
                "Save the notebook, restart the kernel and Run All after editing: " + identifier
            )
    REGISTERED_CORE_CELLS.add(identifier)


def assert_notebook_ready():
    if read_saved_core_sources() != SAVED_CORE_SOURCES:
        raise ValueError(
            "Saved implementation changed during this kernel session. Restart and Run All."
        )
    missing = set(NOTEBOOK_CODE) - REGISTERED_CORE_CELLS
    if missing:
        raise ValueError("Run all definition cells first: " + str(sorted(missing)))


def implementation_hash(identifier):
    return NOTEBOOK_CODE[identifier]


def notify_notebook(event, **payload):
    if _NOTEBOOK_OBSERVER is not None:
        # Display callbacks may not alter training/generation randomness.
        with preserve_rng():
            _NOTEBOOK_OBSERVER(event, payload)


print("Kernel:", sys.executable)
print("Notebook:", NOTEBOOK_PATH)
print("CUDA/PyTorch is checked again before real work. No model is loaded by this cell.")

register_notebook_cell("notebook-runtime")

## 2. Your settings — edit here, then save

`full_path` is the downloaded **TOFU full** model, not the base Meta checkpoint.
`retain_path` is the downloaded TOFU retain95 model. Leave `tokenizer_path`
empty to use the tokenizer beside the full model's `model.safetensors`.
`data_path` holds the six pinned original TOFU `.json` files, not metadata-only
Arrow folders. All paths below can use `C:/...` on Windows.

Keep seeds 42/43/44 for the declared study. `device_micro_batch` may be lowered
to 2 or 1 if needed; logical microbatch 4 × accumulation 8 remains batch 32,
with the same retain-token denominator. Change scientific settings only as a
declared new experiment. Resume requires the same settings/code/packages.

In [ ]:
SETTINGS = {
    "full_path": "",
    "retain_path": "",
    "tokenizer_path": "",
    "data_path": "",
    "output_root": "runs",
    "allow_model_download": True,
    "allow_data_download": True,
    "device": "cuda:0",
    "dtype": "bfloat16",
    "seeds": [42, 43, 44],
    "evaluation_seed": 20261103,
    "beta": 0.1,
    "npo_coefficient": 1.0,
    "retain_coefficient": 1.0,
    "learning_rate": 1e-05,
    "weight_decay": 0.01,
    "nominal_epochs": 10,
    "logical_micro_batch": 4,
    "accumulation": 8,
    "warmup_steps": 6,
    "evaluation_every_updates": 6,
    "training_max_length": 500,
    "evaluation_max_length": 200,
    "retain_sampling": "reshuffled_stream",
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "lora_targets": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "gradient_checkpointing": True,
    "evaluation_retain_count": 300,
    "generation_batch_size": 30,
    "source_temperature": 0.6,
    "source_top_p": 0.9,
    "legacy_greedy_tokens": 96,
    "long_audit_tokens": 256,
    "long_audit_per_split": 20,
    "bootstrap_repeats": 2000,
    "evaluate_kl": True,
    "quantization_modes": ["int8", "nf4"],
    "quantized_controls": True,
    "keep_recent_checkpoints": 2,
    "system_prompt": "You are a helpful assistant.",
    "template_date": "10 Apr 2025",
    "device_micro_batch": 4,
    "generation_eos_ids": [],
}

# Optional interruption recovery. Paste the exact printed run directory.
RESUME_RUN = None  # e.g. r"C:\FYP\runs\20261103_..."
RUN_SELF_TESTS = True
LIVE_PLOT_EVERY_UPDATES = 4
# False lets you inspect definitions/tests without starting the full experiment.
RUN_EXPERIMENT = True

## 3. Configuration and immutable input registry

The following definitions and checkpoint/data/tokenizer hashes are embedded
in this notebook. They do not read a repository config or import local code.

In [ ]:
@dataclass
class Config:
    full_path: str = ""
    retain_path: str = ""
    tokenizer_path: str = ""
    data_path: str = ""
    output_root: str = "runs"
    allow_model_download: bool = False
    allow_data_download: bool = False
    device: str = "cuda:0"
    dtype: str = "bfloat16"
    seeds: list = field(default_factory=lambda: [42, 43, 44])
    evaluation_seed: int = 20261103
    beta: float = 0.1
    npo_coefficient: float = 1.0
    retain_coefficient: float = 1.0
    learning_rate: float = 1e-5
    weight_decay: float = 0.01
    nominal_epochs: int = 10
    logical_micro_batch: int = 4
    device_micro_batch: int = 4
    accumulation: int = 8
    warmup_steps: int = 6
    evaluation_every_updates: int = 6
    training_max_length: int = 500
    evaluation_max_length: int = 200
    retain_sampling: str = "reshuffled_stream"
    lora_rank: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    lora_targets: list = field(
        default_factory=lambda: [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ]
    )
    gradient_checkpointing: bool = True
    evaluation_retain_count: int = 300
    generation_batch_size: int = 30
    source_temperature: float = 0.6
    source_top_p: float = 0.9
    generation_eos_ids: list = field(default_factory=list)
    legacy_greedy_tokens: int = 96
    long_audit_tokens: int = 256
    long_audit_per_split: int = 20
    bootstrap_repeats: int = 2000
    evaluate_kl: bool = True
    quantization_modes: list = field(default_factory=lambda: ["int8", "nf4"])
    quantized_controls: bool = True
    keep_recent_checkpoints: int = 2
    system_prompt: str = "You are a helpful assistant."
    template_date: str = "10 Apr 2025"

    def validate(self):
        numeric = (
            self.beta,
            self.npo_coefficient,
            self.retain_coefficient,
            self.learning_rate,
            self.weight_decay,
            self.lora_dropout,
            self.source_temperature,
            self.source_top_p,
        )
        if any(not math.isfinite(value) for value in numeric):
            raise ValueError("Finite numeric configuration required")
        if self.dtype not in ("bfloat16", "float32"):
            raise ValueError("BF16 for RTX 3060; FP32 is only the CPU fixture path")
        if self.device == "cpu" and self.dtype != "float32":
            raise ValueError("CPU fixtures require float32")
        if not self.seeds or len(set(self.seeds)) != len(self.seeds):
            raise ValueError("Distinct declared training seeds required")
        if any(type(seed) is not int or seed < 0 for seed in self.seeds):
            raise ValueError("Nonnegative integer seeds required")
        if self.retain_sampling not in ("reshuffled_stream", "source_fixed"):
            raise ValueError("Unknown retain sampling mode")
        if self.beta <= 0 or self.npo_coefficient <= 0 or self.retain_coefficient < 0:
            raise ValueError("Invalid objective coefficients")
        if self.lora_rank <= 0 or self.lora_alpha <= 0 or not 0 <= self.lora_dropout < 1:
            raise ValueError("Invalid LoRA configuration")
        if (
            min(
                self.logical_micro_batch,
                self.device_micro_batch,
                self.accumulation,
                self.evaluation_every_updates,
                self.generation_batch_size,
            )
            <= 0
        ):
            raise ValueError("Positive batch, accumulation and evaluation intervals required")
        if self.device_micro_batch > self.logical_micro_batch:
            raise ValueError("Device microbatch cannot exceed the logical microbatch")
        if self.keep_recent_checkpoints < 2:
            raise ValueError("Retain at least two checkpoints for corruption fallback")
        if any(q not in ("int8", "nf4") for q in self.quantization_modes):
            raise ValueError("Unknown quantization mode")
        if self.training_max_length < self.evaluation_max_length or self.evaluation_max_length <= 1:
            raise ValueError("Invalid sequence limits")
        if (
            min(
                self.legacy_greedy_tokens,
                self.long_audit_tokens,
                self.long_audit_per_split,
                self.bootstrap_repeats,
            )
            <= 0
        ):
            raise ValueError("Positive generation/audit/bootstrap sizes required")
        if (
            self.learning_rate <= 0
            or self.weight_decay < 0
            or not 0 < self.source_top_p <= 1
            or self.source_temperature <= 0
        ):
            raise ValueError("Invalid optimizer or sampling parameters")
        return self

    @property
    def effective_batch(self):
        return self.logical_micro_batch * self.accumulation

    def to_dict(self):
        return asdict(self)


PROJECT_LORA_POLICY = {
    "schema": "fq-ssiuu-lora-default-v1",
    "decision": "DL-024",
    "training_mode": "lora",
    "base_quantization_during_training": None,
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "bias": "none",
    "deployment": "save adapter; merge unquantized base; evaluate merge drift; quantize merged "
    "checkpoint",
    "status": "RESEARCHER_SELECTED_IMPLEMENTATION_PENDING_GPU_ACCEPTANCE",
}

register_notebook_cell("configuration")

In [ ]:
PINS = {
    "schema_version": "1.0",
    "status": "PINNED_FOR_METHOD_VALIDATION_ONLY",
    "target": {
        "model_id": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
        "revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
        "checkpoint_sha256": "35b0cd9f7db2735538ff78b4879fab61b23b7ca316df27b059585bf656700a8d",
        "tokenizer_id": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
        "tokenizer_revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
        "config_sha256": "f851a2874b759ee56099c11786b9de7f3e6051677615d4a88259ef0e206ba8a9",
    },
    "retain_only_oracle": {
        "model_id": "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain95",
        "revision": "f05bc9fd6c8d59decf7599f5e186153081061fd2",
        "checkpoint_sha256": "955e430aac277d9ccc309307f8462ce462c81fd4308cd934a6c24e5518077e94",
        "tokenizer_id": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
        "tokenizer_revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
    },
    "dataset": {
        "id": "locuslab/TOFU",
        "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
        "forget": "forget05",
        "retain": "retain95",
        "forget_diagnostic": "forget05_perturbed",
        "utility": ["retain_perturbed", "real_authors_perturbed", "world_facts_perturbed"],
    },
    "files": [
        {
            "repo": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
            "revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
            "path": "config.json",
            "bytes": 927,
            "sha256": "f851a2874b759ee56099c11786b9de7f3e6051677615d4a88259ef0e206ba8a9",
        },
        {
            "repo": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
            "revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
            "path": "tokenizer.json",
            "bytes": 17209920,
            "sha256": "6b9e4e7fb171f92fd137b777cc2714bf87d11576700a1dcd7a399e7bbe39537b",
        },
        {
            "repo": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
            "revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
            "path": "tokenizer_config.json",
            "bytes": 54557,
            "sha256": "df757249013ee916c1318f170d2763cc2272d10d66c5d73a792ce34c2bd8cbb6",
        },
        {
            "repo": "open-unlearning/tofu_Llama-3.2-1B-Instruct_full",
            "revision": "88e31200b97e4c0c04ae0d2f0b591f427046d192",
            "path": "special_tokens_map.json",
            "bytes": 325,
            "sha256": "1b1835caa5b4d70acaa210fa222b0036f1882f9525c4660fd4810fb3e1e40ff8",
        },
        {
            "repo": "locuslab/TOFU",
            "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
            "path": "forget05.json",
            "bytes": 56437,
            "sha256": "88542818f98fd6f880ecaea71bc3f3e7a34f959944e5ff1cdef7ea8c5c137173",
        },
        {
            "repo": "locuslab/TOFU",
            "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
            "path": "retain95.json",
            "bytes": 1021406,
            "sha256": "6625bb8e7e01f27a237bd273b961fcf2fb51568157702acdcd934b13398aa2fc",
        },
        {
            "repo": "locuslab/TOFU",
            "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
            "path": "forget05_perturbed.json",
            "bytes": 334176,
            "sha256": "7596c863a3f7eafb551a167363826f71a7327f28617a0a533fed06108a50ccdb",
        },
        {
            "repo": "locuslab/TOFU",
            "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
            "path": "retain_perturbed.json",
            "bytes": 641057,
            "sha256": "fc69f33bad70d3dca65920bdb54380039e3e8872dd16bbae20c1a839b99533fb",
        },
        {
            "repo": "locuslab/TOFU",
            "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
            "path": "real_authors_perturbed.json",
            "bytes": 17146,
            "sha256": "a23255383ac75a5fbe5acd57622a03fb9c368954fd7b2428afa152b784d9533a",
        },
        {
            "repo": "locuslab/TOFU",
            "revision": "324592d84ae4f482ac7249b9285c2ecdb53e3a68",
            "path": "world_facts_perturbed.json",
            "bytes": 18053,
            "sha256": "0e171838040d0ec94b1ce248d891508e85e3b77344370077caa27b61898eb3e1",
        },
    ],
    "upstreams": {
        "npo": {
            "revision": "1207bbe9a1abf502bc2d7b43a0a467d9524b4036",
            "files": {
                "TOFU/data_module.py": "8e076f18c20b663dbad9862a5d97c25e91fe4f9475bed77e597dbc2b349abf31",
                "TOFU/dataloader.py": "a692b709d01d05204a81e40c38b1f6e64faac9286a662d9d7cf9e4cc4dcf6c69",
            },
        },
        "openunlearning": {
            "revision": "4ad738aaf60f6a4385f6e2506d01da99e76c31f3",
            "files": {
                "src/data/utils.py": "128262e291bd738ffa417099fd076a577494b260297fbf5fd4e317888bcc9bd4",
                "src/data/collators.py": "b7e821c8175e31387c06729a60ff5287a512b681b10379ea21ac72fbff3e786c",
                "configs/model/Llama-3.2-1B-Instruct.yaml": "0d7fea3a3d6820b4cdbd1c672b5dab1f789bcc868a0734af5389bb254fa54fbe",
                "src/evals/metrics/memorization.py": "5d04d81e45ab6eac6fe6ee659f355a1ba57e6809dff860bcb8acebc8be764e9a",
                "src/evals/metrics/privacy.py": "05527a628cc1ba72b2dceeb8e895c5238acbae894243e1bb5d491c280edce03d",
                "src/evals/metrics/utility.py": "2dc70dcf54b06944dddaa3938b8eeceb3e91eb2d706389061fc04bf80a3eaa0e",
                "src/evals/metrics/utils.py": "650f30143228e14cc47b70704b66a4425fe03d8d9cfab0ae315602041bb7d89c",
                "configs/eval/tofu_metrics/model_utility.yaml": "34d95475601379e0f23846239710a18682437e383bdc239f243380c600aaf20c",
            },
        },
    },
}

register_notebook_cell("input-registry")

In [ ]:
CONFIG = Config(**SETTINGS).validate()
for key in ("full_path", "retain_path", "tokenizer_path", "data_path", "output_root"):
    value = getattr(CONFIG, key)
    if value and not Path(value).is_absolute():
        setattr(CONFIG, key, str((NOTEBOOK_HOME / value).resolve()))
display(pd.Series(CONFIG.to_dict(), name="Declared settings").to_frame())

## 4. Metrics and NPO objective

For each forget answer, let \(\ell_\theta=-\log p_\theta(y\mid x)\)
**sum over scored response tokens**. We minimize
\[
L_{\rm NPO}=\frac{2}{\beta}\;\mathbb E\left[
  \operatorname{softplus}\!\left(\beta(\ell_{\rm ref}-\ell_\theta)\right)\right],
\quad L=L_{\rm NPO}+L_{\rm retain\ CE},\quad \beta=0.1.
\]
The full checkpoint is frozen reference; retain95 is the evaluation oracle.
Retain CE is token averaged within the logical four-example group. Geometric
correct-versus-wrong preference is used for FQ; arithmetic wrong-probability
averaging is used for Utility. A KS p-value is not a recovery magnitude.

In [ ]:
def _json_default(value):
    if isinstance(value, set):
        return sorted(value)
    if isinstance(value, (Path, Enum)):
        return value.value if isinstance(value, Enum) else str(value)
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Unsupported metadata type: {type(value).__name__}")


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024**2), b""):
            h.update(block)
    return h.hexdigest()


def digest(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, ensure_ascii=False, default=_json_default).encode()
    ).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False, default=_json_default)
        + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def truth_logs(correct_mean_nll, wrong_mean_nll):
    wrong = np.asarray(wrong_mean_nll, dtype=np.float64)
    if (
        wrong.ndim != 1
        or not len(wrong)
        or not np.isfinite(wrong).all()
        or not np.isfinite(correct_mean_nll)
    ):
        raise ValueError("Finite paraphrased-answer and perturbed-answer NLLs required")
    return {
        "arithmetic_truth_ratio_log": float(
            logsumexp(-wrong) - np.log(len(wrong)) + correct_mean_nll
        ),
        "source_geometric_truth_ratio_log": float(wrong.mean() - correct_mean_nll),
    }


def harmonic(values):
    values = np.asarray(values, dtype=np.float64)
    if not len(values) or not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Complete nonnegative utility components required")
    return 0.0 if (values == 0).any() else float(len(values) / np.reciprocal(values).sum())


def enrich(frame):
    """Recompute definitions from saved primitive NLLs, not old summary fields."""
    frame = frame.copy()
    required = {
        "id",
        "split",
        "question",
        "ground_truth",
        "answer_nll",
        "paraphrase_mean_nll",
        "perturbed_mean_nll",
        "rouge_l_recall",
    }
    if not required.issubset(frame):
        raise ValueError(f"Missing per-question fields: {sorted(required - set(frame))}")
    if frame.duplicated(["split", "id"]).any():
        raise ValueError("Duplicate question IDs")
    computed = []
    for row in frame.to_dict("records"):
        wrong = row["perturbed_mean_nll"]
        if isinstance(wrong, str):
            wrong = ast.literal_eval(wrong)
        logs = truth_logs(row["paraphrase_mean_nll"], wrong)
        mean = float(row["answer_nll"])
        if not np.isfinite(mean) or mean < 0:
            raise ValueError("Nonfinite or negative gold-answer NLL")
        probability = float(np.exp(-mean))
        normalized = float(np.exp(-mean - logsumexp([-mean] + [-float(x) for x in wrong])))
        computed.append(
            {
                **logs,
                "answer_probability": probability,
                "utility_probability": probability
                if row["split"] in ("forget05", "retain")
                else normalized,
                "arithmetic_truth_score": float(
                    -np.expm1(min(0.0, logs["arithmetic_truth_ratio_log"]))
                ),
                "source_geometric_truth_score": float(
                    -np.expm1(min(0.0, -logs["source_geometric_truth_ratio_log"]))
                ),
            }
        )
    for col in computed[0]:
        frame[col] = [row[col] for row in computed]
    return frame


def align(frame, reference, split="forget05"):
    a = frame.loc[frame["split"] == split].set_index("id").sort_index()
    b = reference.loc[reference["split"] == split].set_index("id").sort_index()
    if not len(a) or not a.index.equals(b.index):
        raise ValueError("Matched nonempty question IDs required")
    for col in ("question", "ground_truth"):
        if not a[col].equals(b[col]):
            raise ValueError(f"Mismatched {col}")
    return a, b


def forget_quality(candidate, oracle):
    a, b = align(candidate, oracle)
    x = a["source_geometric_truth_ratio_log"].to_numpy(dtype=float)
    y = b["source_geometric_truth_ratio_log"].to_numpy(dtype=float)
    if not np.isfinite(x).all() or not np.isfinite(y).all():
        raise ValueError("Nonfinite geometric preferences")
    test = ks_2samp(x, y, alternative="two-sided", method="auto")
    arithmetic = ks_2samp(
        a["arithmetic_truth_ratio_log"], b["arithmetic_truth_ratio_log"], method="auto"
    )
    return {
        "forget_quality": float(test.pvalue),
        "ks_statistic": float(test.statistic),
        "paper_arithmetic_ks_statistic_diagnostic": float(arithmetic.statistic),
        "paper_arithmetic_ks_p_value_diagnostic": float(arithmetic.pvalue),
        "n": len(a),
        "definition": "tofu-geometric-preference-KS-v1",
        "interpretation": "Non-rejection is not proof of equivalence or erasure",
    }


def summarize(frame, oracle=None):
    frame = enrich(frame)
    parts = (
        frame.loc[frame["split"] != "forget05"]
        .groupby("split", sort=True)[
            ["utility_probability", "rouge_l_recall", "arithmetic_truth_score"]
        ]
        .mean()
    )
    if set(parts.index) != {"retain", "real_authors", "world_facts"}:
        raise ValueError("All nine TOFU utility components required")
    record = {
        "model_utility": harmonic(parts.to_numpy().flatten()),
        "utility_definition": "tofu-arithmetic-nine-components-v1",
    }
    for split, group in frame.groupby("split", sort=True):
        record[f"{split}_n"] = len(group)
        for col in (
            "answer_probability",
            "answer_nll",
            "rouge_l_recall",
            "token_f1",
            "exact_match",
            "source_geometric_truth_ratio_log",
            "generation_hit_limit",
            "eos_terminated",
            "kl_full_ref_to_model_all_positions",
        ):
            if col in group:
                record[f"{split}_{col}"] = (
                    float(group[col].mean()) if group[col].notna().any() else None
                )
    if oracle is not None:
        record.update(forget_quality(frame, enrich(oracle)))
    return record, parts.reset_index()


def paired_bootstrap(before, after, repeats=2000, seed=42, clusters=None):
    a, b = np.asarray(before, dtype=float), np.asarray(after, dtype=float)
    if (
        a.ndim != 1
        or a.shape != b.shape
        or not len(a)
        or not np.isfinite(a).all()
        or not np.isfinite(b).all()
    ):
        raise ValueError("Aligned finite nonempty observations required")
    d = b - a
    rng = np.random.default_rng(seed)
    estimates = []
    if clusters is not None:
        clusters = np.asarray(clusters)
        if len(clusters) != len(d):
            raise ValueError("Cluster IDs must align with questions")
        unique = np.unique(clusters)
        blocks = [d[clusters == c] for c in unique]
        for _ in range(repeats):
            sample = rng.integers(0, len(blocks), len(blocks))
            estimates.append(float(np.concatenate([blocks[i] for i in sample]).mean()))
        unit = "author-block"
    else:
        for _ in range(repeats):
            estimates.append(float(d[rng.integers(0, len(d), len(d))].mean()))
        unit = "question-exploratory"
    low, high = np.quantile(estimates, [0.025, 0.975])
    return {
        "mean_change": float(d.mean()),
        "ci_low": float(low),
        "ci_high": float(high),
        "fraction_increased": float((d > 0).mean()),
        "fraction_decreased": float((d < 0).mean()),
        "resampling_unit": unit,
        "bootstrap_repeats": repeats,
        "n": len(d),
    }


def quantization_comparisons(stages, repeats=2000, seed=42, author_blocks=None):
    """Both fixed-BF16 and same-precision oracles; directional paired statistics."""
    stages = {name: enrich(frame) for name, frame in stages.items()}
    fq_rows, changes = [], []
    for name, frame in stages.items():
        precision = name.rsplit("_", 1)[-1]
        for policy, oracle_name in (
            ("fixed_bf16", "retain95_bf16"),
            ("matched_precision", f"retain95_{precision}"),
        ):
            if oracle_name in stages:
                fq_rows.append(
                    {
                        "stage": name,
                        "oracle_policy": policy,
                        "oracle_stage": oracle_name,
                        **forget_quality(frame, stages[oracle_name]),
                    }
                )
    if "npo_bf16" in stages:
        for mode in ("int8", "nf4"):
            name = f"npo_{mode}"
            if name not in stages:
                continue
            after, before = align(stages[name], stages["npo_bf16"])
            clusters = (
                [author_blocks[q] for q in before.question] if author_blocks is not None else None
            )
            for metric in (
                "source_geometric_truth_ratio_log",
                "answer_probability",
                "rouge_l_recall",
                "token_f1",
                "exact_match",
            ):
                for grouping in [None, clusters] if clusters is not None else [None]:
                    result = paired_bootstrap(
                        before[metric], after[metric], repeats, seed, grouping
                    )
                    row = {
                        "quantization": mode,
                        "metric": metric,
                        **result,
                        "before": float(before[metric].mean()),
                        "after": float(after[metric].mean()),
                    }
                    for control in ("full", "retain95"):
                        base, quant = f"{control}_bf16", f"{control}_{mode}"
                        if base in stages and quant in stages:
                            c1, c0 = align(stages[quant], stages[base])
                            if not c1.index.equals(before.index):
                                raise ValueError("Control question IDs differ")
                            delta = float((c1[metric] - c0[metric]).mean())
                            row[f"{control}_change"] = delta
                            row[f"npo_minus_{control}_change"] = row["mean_change"] - delta
                    changes.append(row)
    return pd.DataFrame(fq_rows), pd.DataFrame(changes)


def load_existing_stages(run_dir, require_complete=True):
    run_dir = Path(run_dir)
    stages, inventory = {}, []
    for folder in sorted(run_dir.iterdir()):
        meta_path, csv_path = folder / "metadata.json", folder / "per_question.csv"
        if not folder.is_dir() or not meta_path.is_file() or not csv_path.is_file():
            continue
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if require_complete and meta.get("status") != "complete":
            raise ValueError(f"Incomplete stage: {folder.name}")
        for filename, expected in meta.get("artifact_hashes", {}).items():
            path = folder / filename
            if not path.is_file() or sha256(path) != expected:
                raise ValueError(f"Evidence hash mismatch: {path}")
            inventory.append({"path": str(path), "sha256": expected, "bytes": path.stat().st_size})
        stages[folder.name] = enrich(pd.read_csv(csv_path))
    if not stages:
        raise ValueError("No completed per-question CSVs found")
    return stages, inventory


register_notebook_cell("metrics")

In [ ]:
def sequence_nll(logits, labels):
    if logits.ndim != 3 or labels.shape != logits.shape[:2] or labels.dtype != torch.long:
        raise ValueError("Unshifted logits [B,T,V] and int64 labels [B,T] required")
    targets = labels[:, 1:]
    scored = targets != -100
    if not scored.any(dim=1).all():
        raise ValueError("Every sequence needs scored causal answer tokens")
    selected = logits[:, :-1][scored]
    if selected.dtype in (torch.bfloat16, torch.float16):
        selected = selected.float()
    token_nll = functional.cross_entropy(selected, targets[scored], reduction="none")
    return (
        torch.zeros_like(targets, dtype=token_nll.dtype).masked_scatter(scored, token_nll).sum(-1)
    )


def npo_loss(current, reference, beta=0.1):
    if (
        not math.isfinite(beta)
        or beta <= 0
        or current.ndim != 1
        or not current.numel()
        or current.shape != reference.shape
    ):
        raise ValueError("Positive beta and aligned sequence NLLs required")
    if not torch.isfinite(current).all() or not torch.isfinite(reference).all():
        raise ValueError("Finite NLLs required")
    return (2.0 / beta) * functional.softplus(beta * (reference.detach() - current)).mean()


register_notebook_cell("objective")

## 5. Input verification, answer masking and replayable sampling

No silent truncation. Verify model/data/tokenizer bytes before loading.
The 62-update stream covers 1,984 distinct retained rows, not all 3,800.
`source_fixed` is available for the controlled original-pairing comparison.

In [ ]:
def resolve_inputs(config):
    from huggingface_hub import hf_hub_download, snapshot_download
    from transformers import AutoTokenizer

    paths = {}
    for kind, setting, pin_key in (
        ("full", "full_path", "target"),
        ("retain95", "retain_path", "retain_only_oracle"),
    ):
        pin = PINS[pin_key]
        value = getattr(config, setting)
        folder = (
            Path(value)
            if value
            else Path(
                snapshot_download(
                    pin["model_id"],
                    revision=pin["revision"],
                    local_files_only=not config.allow_model_download,
                    allow_patterns=["*.safetensors", "*.json"],
                )
            )
        )
        weight = folder / "model.safetensors"
        if not weight.is_file():
            raise FileNotFoundError(f"Original downloaded weights required: {weight}")
        if sha256(weight) != pin["checkpoint_sha256"]:
            raise ValueError(f"Wrong {kind} model bytes")
        settings = json.loads((folder / "config.json").read_text())
        if settings.get("quantization_config") or settings.get("model_type") != "llama":
            raise ValueError("Original unquantized Llama checkpoint required")
        paths[kind] = folder.resolve()
    if sha256(paths["full"] / "config.json") != PINS["target"]["config_sha256"]:
        raise ValueError("Full-model configuration changed")
    generation_path = paths["full"] / "generation_config.json"
    if (
        sha256(generation_path)
        != "f571455b7715ad274df4ffe832ab20e374e2ea1d6fde467a80e04da04e038f71"
    ):
        raise ValueError("Original full-model generation configuration changed")
    source_generation = json.loads(generation_path.read_text())
    eos = source_generation["eos_token_id"]
    config.generation_eos_ids = eos if isinstance(eos, list) else [eos]
    paths["tokenizer"] = Path(config.tokenizer_path) if config.tokenizer_path else paths["full"]
    tokenizer_files = [
        f
        for f in PINS["files"]
        if f["repo"] == PINS["target"]["model_id"]
        and f["path"].startswith(("tokenizer", "special_tokens"))
    ]
    for item in tokenizer_files:
        if sha256(paths["tokenizer"] / item["path"]) != item["sha256"]:
            raise ValueError(f"Tokenizer differs: {item['path']}")
    tokenizer = AutoTokenizer.from_pretrained(
        str(paths["tokenizer"]), local_files_only=True, trust_remote_code=False
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    data, receipts = {}, {}
    for item in PINS["files"]:
        if item["repo"] != PINS["dataset"]["id"]:
            continue
        path = (
            Path(config.data_path) / item["path"] if config.data_path else Path("__not_supplied__")
        )
        if not path.is_file():
            path = Path(
                hf_hub_download(
                    PINS["dataset"]["id"],
                    filename=item["path"],
                    repo_type="dataset",
                    revision=PINS["dataset"]["revision"],
                    local_files_only=not config.allow_data_download,
                )
            )
        if sha256(path) != item["sha256"]:
            raise ValueError(f"Dataset bytes changed: {path}")
        name = Path(item["path"]).stem
        data[name] = [
            json.loads(line)
            for line in path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        receipts[name] = {"sha256": item["sha256"], "path": str(path), "rows": len(data[name])}
    if len(data["forget05"]) != 200 or len(data["retain95"]) != 3800:
        raise ValueError("Expected complete forget05 and retain95 splits")
    return paths, tokenizer, data, receipts


def prompt_ids(tokenizer, question, config):
    return list(
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": config.system_prompt},
                {"role": "user", "content": question},
            ],
            tokenize=True,
            return_dict=False,
            add_generation_prompt=True,
            date_string=config.template_date,
        )
    )


def encode_qa(tokenizer, question, answer, config, max_length=None):
    if (
        not isinstance(question, str)
        or not question.strip()
        or not isinstance(answer, str)
        or not answer.strip()
    ):
        raise ValueError("Nonempty question and answer required")
    prompt = prompt_ids(tokenizer, question, config)
    ids = list(
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": config.system_prompt},
                {"role": "user", "content": question},
                {"role": "assistant", "content": answer},
            ],
            tokenize=True,
            return_dict=False,
            add_generation_prompt=False,
            date_string=config.template_date,
        )
    )
    if ids[: len(prompt)] != prompt:
        raise ValueError("Prompt is not an exact token prefix")
    if ids[-1] != tokenizer.eos_token_id:
        ids.append(tokenizer.eos_token_id)
    limit = max_length or config.training_max_length
    if len(ids) > limit:
        raise ValueError(
            f"Unexpected truncation would be required ({len(ids)} > {limit}): {question}"
        )
    if len(ids) <= len(prompt) + 1:
        raise ValueError("No answer content")
    return {
        "input_ids": ids,
        "labels": [-100] * len(prompt) + ids[len(prompt) :],
        "prompt_length": len(prompt),
    }


def one_batch(example, device, pad_id, fixed_length=None):
    ids, labels = list(example["input_ids"]), list(example["labels"])
    length = len(ids)
    target_length = fixed_length or length
    if target_length < length:
        raise ValueError("Truncation forbidden")
    mask = [1] * length + [0] * (target_length - length)
    ids += [pad_id] * (target_length - length)
    labels += [-100] * (target_length - length)
    return {
        "input_ids": torch.tensor([ids], device=device),
        "attention_mask": torch.tensor([mask], device=device),
    }, torch.tensor([labels], device=device)


def prepare_examples(data, tokenizer, config):
    forget_rows, retain_rows = data["forget05"], data["retain95"]
    forget_q = {r["question"] for r in forget_rows}
    retain_q = {r["question"] for r in retain_rows}
    if forget_q & retain_q:
        raise ValueError("Forget/retain leakage")
    forget = [encode_qa(tokenizer, r["question"], r["answer"], config) for r in forget_rows]
    retain = [encode_qa(tokenizer, r["question"], r["answer"], config) for r in retain_rows]
    if {r["question"] for r in data["forget05_perturbed"]} != forget_q:
        raise ValueError("Forget diagnostic differs from training split")
    eligible = [(i, r) for i, r in enumerate(data["retain_perturbed"]) if r["question"] in retain_q]
    rng = np.random.default_rng(config.evaluation_seed)
    chosen = sorted(
        rng.choice(len(eligible), config.evaluation_retain_count, replace=False).tolist()
    )
    selected = {
        "forget05": list(enumerate(data["forget05_perturbed"])),
        "retain": [eligible[i] for i in chosen],
        "real_authors": list(enumerate(data["real_authors_perturbed"])),
        "world_facts": list(enumerate(data["world_facts_perturbed"])),
    }
    source_indices = {row["question"]: i for i, row in enumerate(forget_rows)}
    packs = {}
    for split, rows in selected.items():
        packs[split] = []
        for source_index, row in rows:
            base = row["paraphrased_answer"] if split in ("forget05", "retain") else row["answer"]
            pack = {
                "id": f"{split}:{source_index:04d}",
                "split": split,
                "source_index": source_index,
                "question": row["question"],
                "ground_truth": row["answer"],
                "gold": encode_qa(
                    tokenizer, row["question"], row["answer"], config, config.evaluation_max_length
                ),
                "base": encode_qa(
                    tokenizer, row["question"], base, config, config.evaluation_max_length
                ),
                "wrong": [
                    encode_qa(tokenizer, row["question"], x, config, config.evaluation_max_length)
                    for x in row["perturbed_answer"]
                ],
            }
            if split == "forget05":
                # The pinned TOFU split stores 20 contiguous QAs per fictitious author.
                pack["author_block"] = source_indices[row["question"]] // 20
            packs[split].append(pack)
    selection = {
        split: [
            {"id": p["id"], "source_index": p["source_index"], "question": p["question"]}
            for p in rows
        ]
        for split, rows in packs.items()
    }
    return forget, retain, packs, selection


def build_plan(forget_count, retain_count, config, seed):
    steps = forget_count * config.nominal_epochs // config.effective_batch
    if steps <= config.warmup_steps or forget_count % config.logical_micro_batch:
        raise ValueError("Unsupported schedule or incomplete logical microbatch")
    generator = torch.Generator().manual_seed(1001 + seed - config.seeds[0])
    loader = DataLoader(
        list(range(forget_count)),
        batch_size=config.logical_micro_batch,
        shuffle=True,
        generator=generator,
        num_workers=0,
        drop_last=False,
    )
    batches = []
    while len(batches) < steps * config.accumulation:
        for batch in loader:
            batches.append(batch.tolist())
            if len(batches) == steps * config.accumulation:
                break
    retain_generator = torch.Generator().manual_seed(seed + 2001)
    pool, cursor, retain_batches = [], 0, []
    for batch in batches:
        paired = []
        for index in batch:
            if config.retain_sampling == "source_fixed":
                draw = torch.randint(
                    retain_count, (1,), generator=torch.Generator().manual_seed(index)
                ).item()
                paired.append((index + draw) % retain_count)
            else:
                if cursor == len(pool):
                    pool = torch.randperm(retain_count, generator=retain_generator).tolist()
                    cursor = 0
                paired.append(pool[cursor])
                cursor += 1
        retain_batches.append(paired)
    return {
        "steps": steps,
        "forget_batches": batches,
        "retain_batches": retain_batches,
        "evaluation_steps": sorted(
            set(range(config.evaluation_every_updates, steps + 1, config.evaluation_every_updates))
            | {steps}
        ),
        "unique_retain_rows": len({i for batch in retain_batches for i in batch}),
        "retain_sampling": config.retain_sampling,
        "effective_data_passes": steps * config.effective_batch / forget_count,
        "seed": seed,
    }


register_notebook_cell("data")

## 6. Model loading, likelihoods, generation and provenance

Evaluation produces actual answers and token IDs as it runs. Primary sampled
generation uses total length 200/batch30. Every final stage also gets greedy96
on all questions and a frozen 80-question greedy256 audit. Partial outputs are
preserved; completed results are reused only if their identities and hashes match.

In [ ]:
def release_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def dtype_for(config):
    return getattr(torch, config.dtype)


def model_loader(path, config, quantization=None):
    major = int(importlib.metadata.version("transformers").split(".")[0])
    kwargs = {
        "local_files_only": True,
        "trust_remote_code": False,
        "use_safetensors": True,
        "low_cpu_mem_usage": True,
        "attn_implementation": "sdpa",
        ("dtype" if major >= 5 else "torch_dtype"): dtype_for(config),
    }
    if quantization:
        if not str(config.device).startswith("cuda"):
            raise ValueError("This experiment's bitsandbytes path requires CUDA")
        if quantization == "int8":
            qc = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
        elif quantization == "nf4":
            qc = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=dtype_for(config),
                bnb_4bit_use_double_quant=True,
            )
        else:
            raise ValueError(quantization)
        kwargs.update(quantization_config=qc, device_map={"": str(config.device)})
    model = AutoModelForCausalLM.from_pretrained(str(path), **kwargs)
    if quantization is None:
        model.to(config.device)
    model.requires_grad_(False).eval()
    if quantization:
        import bitsandbytes as bnb

        kind = bnb.nn.Linear8bitLt if quantization == "int8" else bnb.nn.Linear4bit
        count = sum(isinstance(m, kind) for m in model.modules())
        loaded = getattr(
            model, "is_loaded_in_8bit" if quantization == "int8" else "is_loaded_in_4bit", False
        )
        if not count or not loaded:
            raise ValueError("Actual quantization was not established")
    return model


def tensor_fingerprint(model, skip_adapters=False):
    """Hash packed weights and quantization statistics without a whole CPU copy."""
    records = []
    for name, tensor in sorted(model.state_dict().items()):
        if skip_adapters and "lora_" in name:
            continue
        if not isinstance(tensor, torch.Tensor):
            records.append({"name": name, "metadata": str(tensor)})
            continue
        h = __import__("hashlib").sha256()
        flat = tensor.detach().reshape(-1)
        for start in range(0, flat.numel(), 1_048_576):
            raw = (
                flat[start : start + 1_048_576]
                .cpu()
                .contiguous()
                .view(torch.uint8)
                .numpy()
                .tobytes()
            )
            h.update(raw)
        records.append(
            {
                "name": name,
                "shape": list(tensor.shape),
                "dtype": str(tensor.dtype),
                "sha256": h.hexdigest(),
            }
        )
    return {"sha256": digest(records), "tensors": records}


def model_metadata(model, quantization):
    qc = getattr(model.config, "quantization_config", None)
    if qc is not None and hasattr(qc, "to_dict"):
        qc = qc.to_dict()
    modules = []
    if quantization:
        import bitsandbytes as bnb

        kind = bnb.nn.Linear8bitLt if quantization == "int8" else bnb.nn.Linear4bit
        modules = [name for name, layer in model.named_modules() if isinstance(layer, kind)]
    return {
        "quantization": quantization,
        "quantization_config": qc,
        "quantized_linear_modules": modules,
        "quantized_module_count": len(modules),
        "model_memory_bytes": model.get_memory_footprint(),
        "packed_state": tensor_fingerprint(model),
    }


@contextmanager
def preserve_rng():
    cpu, cuda = (
        torch.get_rng_state(),
        torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    )
    python, numpy = random.getstate(), np.random.get_state()
    try:
        yield
    finally:
        random.setstate(python)
        np.random.set_state(numpy)
        torch.set_rng_state(cpu)
        if cuda:
            torch.cuda.set_rng_state_all(cuda)


def text_scores(gold, generated):
    def normalize(text):
        return " ".join(unicodedata.normalize("NFKC", text).casefold().split())

    a, b = re.findall(r"\w+", normalize(gold)), re.findall(r"\w+", normalize(generated))
    common = sum((Counter(a) & Counter(b)).values())
    return {
        "rouge_l_recall": float(
            rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
            .score(gold, generated)["rougeL"]
            .recall
        ),
        "token_f1": 2 * common / (len(a) + len(b)) if a and b else 0.0,
        "exact_match": float(normalize(gold) == normalize(generated)),
    }


@torch.no_grad()
def score_example(model, example, tokenizer, config, reference=None):
    inputs, labels = one_batch(
        example, config.device, tokenizer.pad_token_id, config.evaluation_max_length
    )
    logits = model(**inputs, use_cache=False).logits
    nll = sequence_nll(logits, labels)
    mask = labels[:, 1:] != -100
    targets = labels[:, 1:][mask].unsqueeze(1)
    tokens = (
        functional.log_softmax(logits[:, :-1][mask].float(), dim=-1).gather(1, targets).flatten()
    )
    values = tokens.cpu().double().tolist()
    kl = None
    if reference is model:
        kl = 0.0
    elif reference is not None:
        ref_logits = reference(**inputs, use_cache=False).logits
        total = 0.0
        for start in range(0, logits.shape[1], 8):
            current = logits[:, start : start + 8].float().log_softmax(-1)
            frozen = ref_logits[:, start : start + 8].float().log_softmax(-1)
            total += float(
                functional.kl_div(current, frozen, log_target=True, reduction="none")
                .sum(-1)
                .double()
                .sum()
            )
        kl = total / logits.shape[1]
        del ref_logits
    count, total = int(mask.sum()), float(nll.item())
    if not math.isfinite(total) or (kl is not None and not math.isfinite(kl)):
        raise FloatingPointError("Nonfinite evaluation score")
    k = max(1, math.ceil(len(values) * 0.2))
    return {
        "sequence_nll": total,
        "scored_tokens": count,
        "answer_nll": total / count,
        "token_log_probabilities": values,
        "min_k20_log_probability": float(np.mean(sorted(values)[:k])),
        "kl_full_ref_to_model_all_positions": kl,
    }


@torch.no_grad()
def generate_batch(model, rows, tokenizer, config, profile):
    prompts = [prompt_ids(tokenizer, p["question"], config) for p in rows]
    width = max(map(len, prompts))
    ids = torch.full(
        (len(rows), width), tokenizer.pad_token_id, dtype=torch.long, device=config.device
    )
    masks = torch.zeros_like(ids)
    for i, prompt in enumerate(prompts):
        ids[i, -len(prompt) :] = torch.tensor(prompt, device=config.device)
        masks[i, -len(prompt) :] = 1
    eos = {tokenizer.eos_token_id}
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    if isinstance(eot, int) and eot != tokenizer.unk_token_id:
        eos.add(eot)
    if profile != "legacy_greedy96":
        eos.update(config.generation_eos_ids)
    kwargs = dict(
        num_beams=1,
        eos_token_id=sorted(eos),
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,
        use_cache=True,
        repetition_penalty=1.0,
    )
    if profile == "source_sampled":
        if width >= config.evaluation_max_length:
            raise ValueError("Source generation limit leaves no answer space")
        kwargs.update(
            do_sample=True,
            temperature=config.source_temperature,
            top_p=config.source_top_p,
            max_length=config.evaluation_max_length,
        )
        available = config.evaluation_max_length - width
    else:
        available = (
            config.legacy_greedy_tokens
            if profile == "legacy_greedy96"
            else config.long_audit_tokens
        )
        kwargs.update(do_sample=False, max_new_tokens=available)
    generation = GenerationConfig(**kwargs)
    old = model.generation_config
    try:
        model.generation_config = generation
        torch.manual_seed(0)  # Author's generation seed per batch, isolated from training RNG.
        outputs = model.generate(input_ids=ids, attention_mask=masks, generation_config=generation)
    finally:
        model.generation_config = old
    if not torch.equal(outputs[:, :width], ids):
        raise ValueError("Generation lost the prompt prefix")
    result = []
    for output in outputs[:, width:].cpu().tolist():
        end = next((i for i, token in enumerate(output) if token in eos), None)
        tokens = output[: end + 1] if end is not None else output
        text = tokenizer.decode(
            tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        result.append(
            {
                "generated_answer": text,
                "generated_token_ids": tokens,
                "generated_token_count": len(tokens),
                "eos_terminated": end is not None,
                "generation_hit_limit": end is None and len(tokens) >= available,
                "generation_profile": profile,
                "generation_prompt_width": width,
                "generation_available_tokens": available,
                "generation_eos_token_ids": sorted(eos),
            }
        )
    return result


def _write_frame(folder, frame, oracle=None):
    frame.to_csv(folder / "per_question.csv", index=False, encoding="utf-8-sig")
    matched_oracle = oracle.loc[oracle.id.isin(frame.id)] if oracle is not None else None
    summary, components = summarize(frame, matched_oracle)
    write_json(folder / "summary.json", summary)
    components.to_csv(folder / "utility_components.csv", index=False)
    columns = [
        c
        for c in (
            "answer_nll",
            "answer_probability",
            "rouge_l_recall",
            "token_f1",
            "exact_match",
            "source_geometric_truth_ratio_log",
            "eos_terminated",
            "generation_hit_limit",
            "kl_full_ref_to_model_all_positions",
        )
        if c in frame and frame[c].notna().any()
    ]
    frame.groupby("split")[columns].mean().to_csv(folder / "metrics.csv")
    return summary


def evaluate(
    model,
    tokenizer,
    packs,
    config,
    folder,
    binding,
    reference=None,
    oracle=None,
    profiles=("source_sampled",),
    shared_likelihoods=None,
):
    """Reuse only complete identity-bound stages; partial files remain intact."""
    folder = Path(folder)
    identity = digest(
        {
            "binding": binding,
            "packs": packs,
            "profiles": list(profiles),
            "generation": {
                k: getattr(config, k)
                for k in (
                    "generation_batch_size",
                    "source_temperature",
                    "source_top_p",
                    "evaluation_max_length",
                    "legacy_greedy_tokens",
                    "long_audit_tokens",
                    "long_audit_per_split",
                )
            },
            "evaluator_sha256": implementation_hash("evaluation"),
            "metrics_sha256": implementation_hash("metrics"),
        }
    )
    marker = folder / "metadata.json"
    if marker.is_file():
        old = json.loads(marker.read_text())
        if old.get("status") == "complete":
            if old["identity"] != identity:
                raise ValueError(f"Cached evaluation identity differs: {folder}")
            for name, expected in old["artifact_hashes"].items():
                if sha256(folder / name) != expected:
                    raise ValueError(f"Cached evaluation bytes changed: {folder / name}")
            return pd.read_json(folder / "per_question.jsonl", lines=True)
    if folder.exists():
        folder.rename(folder.with_name(folder.name + f"_incomplete_{time.time_ns()}"))
    folder.mkdir(parents=True)
    previous_mode = model.training
    model.eval()
    started = time.perf_counter()
    metadata = {
        "status": "running",
        "identity": identity,
        "binding": binding,
        "profiles": list(profiles),
        "likelihood_source": "shared" if shared_likelihoods else "measured",
    }
    if str(config.device).startswith("cuda"):
        torch.cuda.reset_peak_memory_stats(config.device)
        free, total = torch.cuda.mem_get_info(config.device)
        metadata.update(device_free_bytes_before=free, device_total_bytes=total)
    write_json(marker, metadata)
    primary = []
    try:
        with preserve_rng():
            scores = shared_likelihoods if shared_likelihoods is not None else {}
            if shared_likelihoods is None:
                with (folder / "likelihoods.jsonl").open("x", encoding="utf-8") as sink:
                    for split, examples in packs.items():
                        print(f"Scoring {folder.name}: {split} ({len(examples)})", flush=True)
                        for p in examples:
                            gold = score_example(model, p["gold"], tokenizer, config, reference)
                            base = (
                                gold["answer_nll"]
                                if p["base"] == p["gold"]
                                else score_example(model, p["base"], tokenizer, config)[
                                    "answer_nll"
                                ]
                            )
                            wrong = [
                                score_example(model, e, tokenizer, config)["answer_nll"]
                                for e in p["wrong"]
                            ]
                            record = {
                                **{
                                    k: p[k]
                                    for k in (
                                        "id",
                                        "split",
                                        "source_index",
                                        "question",
                                        "ground_truth",
                                    )
                                },
                                **gold,
                                "paraphrase_mean_nll": base,
                                "perturbed_mean_nll": wrong,
                            }
                            if "author_block" in p:
                                record["author_block"] = p["author_block"]
                            scores[p["id"]] = record
                            sink.write(
                                json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n"
                            )
                            sink.flush()
            for profile in profiles:
                destination = folder if profile == "source_sampled" else folder / profile
                destination.mkdir(exist_ok=True)
                rows = []
                with (destination / "per_question.jsonl").open("x", encoding="utf-8") as sink:
                    for split, examples in packs.items():
                        selected = examples
                        if profile == "long_greedy":
                            rng = np.random.default_rng(config.evaluation_seed)
                            ix = sorted(
                                rng.choice(
                                    len(examples),
                                    min(len(examples), config.long_audit_per_split),
                                    replace=False,
                                )
                            )
                            selected = [examples[int(i)] for i in ix]
                        print(
                            f"Generating {folder.name}/{profile}: {split} ({len(selected)})",
                            flush=True,
                        )
                        batch_size = (
                            1 if profile == "legacy_greedy96" else config.generation_batch_size
                        )
                        for start in range(0, len(selected), batch_size):
                            batch = selected[start : start + batch_size]
                            generated = generate_batch(model, batch, tokenizer, config, profile)
                            for pack, response in zip(batch, generated):
                                row = {
                                    **scores[pack["id"]],
                                    **response,
                                    **text_scores(
                                        pack["ground_truth"], response["generated_answer"]
                                    ),
                                }
                                rows.append(row)
                                sink.write(
                                    json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n"
                                )
                            sink.flush()
                frame = enrich(pd.DataFrame(rows))
                summary = _write_frame(destination, frame, oracle)
                print(
                    json.dumps(
                        {
                            "profile": profile,
                            "model_utility": summary["model_utility"],
                            "forget_quality": summary.get("forget_quality"),
                        }
                    ),
                    flush=True,
                )
                if profile == "source_sampled":
                    primary = frame
        outputs = [p for p in folder.rglob("*") if p.is_file() and p.name != "metadata.json"]
        metadata.update(
            status="complete",
            seconds=time.perf_counter() - started,
            process_ram_bytes=psutil.Process().memory_info().rss,
            artifact_hashes={str(p.relative_to(folder)): sha256(p) for p in outputs},
        )
        if str(config.device).startswith("cuda"):
            torch.cuda.synchronize(config.device)
            metadata.update(
                peak_torch_allocated_bytes=torch.cuda.max_memory_allocated(config.device),
                peak_torch_reserved_bytes=torch.cuda.max_memory_reserved(config.device),
            )
        write_json(marker, metadata)
        return primary
    except BaseException as exc:
        write_json(
            folder / "failure.json", {"error": repr(exc), "seconds": time.perf_counter() - started}
        )
        raise
    finally:
        model.train(previous_mode)


register_notebook_cell("evaluation")

## 7. Recoverable LoRA training

Only FP32 adapter parameters and AdamW states train over the BF16 frozen base.
Reference forwards disable adapters under eval/no_grad. Every update saves
adapter/optimizer/RNG/history state atomically; retain two complete generations
for corruption fallback. Training tensors remain on the selected CUDA device.

In [ ]:
def capture_rng():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "cpu": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


def restore_rng(rng):
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["cpu"])
    if rng["cuda"]:
        torch.cuda.set_rng_state_all(rng["cuda"])


def _cpu_tree(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {k: _cpu_tree(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_cpu_tree(x) for x in value]
    if isinstance(value, tuple):
        return tuple(_cpu_tree(x) for x in value)
    return value


def _flush_directory(path):
    if os.name != "nt":
        fd = os.open(path, os.O_RDONLY | os.O_DIRECTORY)
        try:
            os.fsync(fd)
        finally:
            os.close(fd)


def checkpoint_candidates(root):
    found = []
    for path in Path(root).glob("step-*"):
        marker = path / "complete.json"
        if marker.is_file():
            try:
                meta = json.loads(marker.read_text())
                found.append((meta["step"], path, meta))
            except (ValueError, KeyError):
                continue
    return sorted(found, key=lambda row: (row[0], row[1].name), reverse=True)


def save_checkpoint(root, model, optimizer, step, identity, history, keep=2):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    path = root / f"step-{step:06d}-{time.time_ns()}"
    pending = root / (".pending-" + path.name)
    pending.mkdir()
    model.save_pretrained(pending / "adapter", safe_serialization=True)
    # Only our own checksummed runtime checkpoints are loaded, never HF training_args.bin.
    torch.save(
        {"optimizer": _cpu_tree(optimizer.state_dict()), "rng": capture_rng(), "history": history},
        pending / "runtime.pt",
    )
    files = {str(p.relative_to(pending)): sha256(p) for p in pending.rglob("*") if p.is_file()}
    for name in files:
        with (pending / name).open("r+b") as handle:
            os.fsync(handle.fileno())
    write_json(
        pending / "complete.json", {"step": step, "identity": identity, "artifact_hashes": files}
    )
    with (pending / "complete.json").open("r+b") as handle:
        os.fsync(handle.fileno())
    pending.rename(path)
    _flush_directory(root)
    for _, old, _ in checkpoint_candidates(root)[keep:]:
        shutil.rmtree(old)  # Only completed generations in this run's owned checkpoint directory.
    return path


def restore_checkpoint(root, model, optimizer, identity):
    from safetensors.torch import load_file

    rejected = []
    for step, path, meta in checkpoint_candidates(root):
        if meta.get("identity") != identity:
            rejected.append({"path": str(path), "reason": "identity mismatch"})
            continue
        try:
            for name, expected in meta["artifact_hashes"].items():
                if sha256(path / name) != expected:
                    raise ValueError(f"Checksum mismatch: {name}")
            state = load_file(str(path / "adapter/adapter_model.safetensors"))
            # weights_only=False is confined to our identity-bound, checksummed local state.
            runtime = torch.load(path / "runtime.pt", map_location="cpu", weights_only=False)
            if [row["update"] for row in runtime["history"]] != list(range(1, step + 1)):
                raise ValueError("Checkpoint history/cursor mismatch")
            keys = set_peft_model_state_dict(model, state)
            if keys.unexpected_keys or any("lora_" in key for key in keys.missing_keys):
                raise ValueError("Checkpoint adapter keys differ")
            optimizer.load_state_dict(runtime["optimizer"])
            restore_rng(runtime["rng"])
            return step, runtime["history"], rejected
        except (OSError, ValueError, RuntimeError, EOFError, KeyError, TypeError) as exc:
            rejected.append({"path": str(path), "reason": str(exc)})
    raise ValueError(f"No compatible complete LoRA checkpoint: {rejected}")


register_notebook_cell("checkpoint")

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def attach_lora(base, config, seed):
    seed_everything(seed)
    base.requires_grad_(False)
    adapter = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config.lora_targets,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(base, adapter, autocast_adapter_dtype=True)
    trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    if not trainable or any("lora_" not in n for n, _ in trainable):
        raise ValueError("Only LoRA parameters may be trainable")
    # Preserve small AdamW updates even when the frozen backbone uses BF16.
    for _, parameter in trainable:
        parameter.data = parameter.data.float()
    return model


def collate(examples, tokenizer, config):
    width = max(len(e["input_ids"]) for e in examples)
    ids = torch.full(
        (len(examples), width), tokenizer.pad_token_id, dtype=torch.long, device=config.device
    )
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    for i, row in enumerate(examples):
        size = len(row["input_ids"])
        ids[i, :size] = torch.tensor(row["input_ids"], device=config.device)
        labels[i, :size] = torch.tensor(row["labels"], device=config.device)
        mask[i, :size] = 1
    return {"input_ids": ids, "attention_mask": mask}, labels


def forward_sequence(model, examples, tokenizer, config):
    inputs, labels = collate(examples, tokenizer, config)
    logits = model(**inputs, use_cache=False).logits
    return sequence_nll(logits, labels), int((labels[:, 1:] != -100).sum())


def frozen_reference_nll(model, examples, tokenizer, config):
    """Adapters off + all base weights frozen = the original unquantized full model."""
    previous = model.training
    try:
        model.eval()
        with model.disable_adapter(), torch.no_grad():
            result, _ = forward_sequence(model, examples, tokenizer, config)
        if result.requires_grad:
            raise ValueError("Reference has a gradient graph")
        return result.detach()
    finally:
        model.train(previous)


def learning_rate(step, total, config):
    if step < config.warmup_steps:
        return config.learning_rate * step / max(1, config.warmup_steps)
    return config.learning_rate * max(0.0, (total - step) / max(1, total - config.warmup_steps))


def train(
    model,
    tokenizer,
    forget,
    retain,
    config,
    seed,
    folder,
    binding,
    evaluate_callback=None,
    resume=False,
    stop_after=None,
):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    plan = build_plan(len(forget), len(retain), config, seed)
    identity = digest(
        {
            "binding": binding,
            "config": config.to_dict(),
            "plan": plan,
            "training_code": implementation_hash("training"),
            "objective_code": implementation_hash("objective"),
            "forget": forget,
            "retain": retain,
        }
    )
    write_json(folder / "training_plan.json", plan)
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    if any("lora_" not in name for name, _ in named) or not named:
        raise ValueError("Full-weight training is forbidden in this LoRA experiment")
    groups = [
        {
            "params": [p for n, p in named if p.ndim >= 2 and not n.endswith("bias")],
            "weight_decay": config.weight_decay,
        },
        {"params": [p for n, p in named if p.ndim < 2 or n.endswith("bias")], "weight_decay": 0.0},
    ]
    optimizer = torch.optim.AdamW(
        groups, lr=config.learning_rate, betas=(0.9, 0.999), eps=1e-8, foreach=False
    )
    base_before = tensor_fingerprint(model, skip_adapters=True)["sha256"]
    history, first_step = [], 0
    checkpoint_root = folder / "checkpoints"
    if resume:
        first_step, history, rejected = restore_checkpoint(
            checkpoint_root, model, optimizer, identity
        )
        write_json(
            folder / "resume_receipt.json",
            {"next_update": first_step + 1, "rejected": rejected, "identity": identity},
        )
    elif list(checkpoint_root.glob("step-*")):
        raise ValueError("Training checkpoints exist; explicitly resume this run")
    write_json(
        folder / "training_identity.json",
        {
            "identity": identity,
            "base_tensor_sha256": base_before,
            "trainable_names": [name for name, _ in named],
            "trainable_parameters": sum(p.numel() for _, p in named),
            "total_parameters": sum(p.numel() for p in model.parameters()),
            "lora": model.peft_config["default"].to_dict(),
            "reference": "same frozen unquantized full backbone with adapters disabled, eval, no_grad",
        },
    )
    model.config.use_cache = False
    if config.gradient_checkpointing:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()
    optimizer.zero_grad(set_to_none=True)
    try:
        if evaluate_callback and first_step and first_step in plan["evaluation_steps"]:
            # A power cut may have followed the checkpoint but preceded its evaluation.
            evaluate_callback(model, first_step, identity)
        for step in range(first_step, plan["steps"]):
            started = time.perf_counter()
            if str(config.device).startswith("cuda"):
                torch.cuda.reset_peak_memory_stats(config.device)
            forget_loss, retain_loss = 0.0, 0.0
            for index in range(step * config.accumulation, (step + 1) * config.accumulation):
                forget_ids, retain_ids = (
                    plan["forget_batches"][index],
                    plan["retain_batches"][index],
                )
                retain_tokens = sum(
                    sum(x != -100 for x in retain[i]["labels"][1:]) for i in retain_ids
                )
                for start in range(0, len(forget_ids), config.device_micro_batch):
                    subset = [
                        forget[i] for i in forget_ids[start : start + config.device_micro_batch]
                    ]
                    reference = frozen_reference_nll(model, subset, tokenizer, config)
                    current, _ = forward_sequence(model, subset, tokenizer, config)
                    term = npo_loss(current, reference, config.beta)
                    weight = len(subset) / (len(forget_ids) * config.accumulation)
                    (config.npo_coefficient * term * weight).backward()
                    forget_loss += float(term.detach()) * weight
                    del current, reference, term
                if config.retain_coefficient:
                    for start in range(0, len(retain_ids), config.device_micro_batch):
                        subset = [
                            retain[i] for i in retain_ids[start : start + config.device_micro_batch]
                        ]
                        nll, _ = forward_sequence(model, subset, tokenizer, config)
                        term = nll.sum() / retain_tokens
                        (config.retain_coefficient * term / config.accumulation).backward()
                        retain_loss += float(term.detach()) / config.accumulation
                        del nll, term
            for name, parameter in named:
                if parameter.grad is None or not torch.isfinite(parameter.grad).all():
                    raise FloatingPointError(f"Missing or nonfinite LoRA gradient: {name}")
            lr = learning_rate(step, plan["steps"], config)
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.step()  # No clipping, matching source effective clipping=0.
            optimizer.zero_grad(set_to_none=True)
            if str(config.device).startswith("cuda"):
                torch.cuda.synchronize(config.device)
            record = {
                "update": step + 1,
                "lr": lr,
                "npo_loss": forget_loss,
                "retain_ce": retain_loss,
                "total_loss": config.npo_coefficient * forget_loss
                + config.retain_coefficient * retain_loss,
                "effective_forget_passes": (step + 1) * config.effective_batch / len(forget),
                "compute_seconds": time.perf_counter() - started,
                "peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(config.device)
                if str(config.device).startswith("cuda")
                else 0,
                "peak_cuda_reserved_bytes": torch.cuda.max_memory_reserved(config.device)
                if str(config.device).startswith("cuda")
                else 0,
                "process_ram_bytes": psutil.Process().memory_info().rss,
            }
            if not all(math.isfinite(value) for value in record.values()):
                raise FloatingPointError("Nonfinite training record")
            history.append(record)
            checkpoint = save_checkpoint(
                checkpoint_root,
                model,
                optimizer,
                step + 1,
                identity,
                history,
                config.keep_recent_checkpoints,
            )
            pd.DataFrame(history).to_csv(folder / "training_log.csv", index=False)
            print(json.dumps({"seed": seed, **record, "checkpoint": str(checkpoint)}), flush=True)
            notify_notebook("training_update", seed=seed, **record)
            if evaluate_callback and step + 1 in plan["evaluation_steps"]:
                evaluate_callback(model, step + 1, identity)
            if stop_after is not None and step + 1 >= stop_after:
                return {"status": "interrupted-fixture", "step": step + 1, "identity": identity}
        base_after = tensor_fingerprint(model, skip_adapters=True)["sha256"]
        if base_before != base_after:
            raise ValueError("Frozen backbone changed during LoRA training")
        model.eval()
        model.config.use_cache = True
        if config.gradient_checkpointing:
            model.gradient_checkpointing_disable()
        adapter_dir = folder / "adapter"
        model.save_pretrained(adapter_dir, safe_serialization=True)
        tokenizer.save_pretrained(adapter_dir)
        receipt = {
            "status": "complete",
            "identity": identity,
            "updates": plan["steps"],
            "base_unchanged": True,
            "base_tensor_sha256": base_after,
            "seed": seed,
            "adapter_hashes": {p.name: sha256(p) for p in adapter_dir.glob("*") if p.is_file()},
        }
        write_json(folder / "training_complete.json", receipt)
        return receipt
    except BaseException as exc:
        write_json(
            folder / "training_failure.json",
            {
                "error": repr(exc),
                "identity": identity,
                "completed_updates": len(history),
                "instruction": "Resume this run; the latest complete adapter/optimizer/RNG checkpoint is authoritative.",
            },
        )
        raise


register_notebook_cell("training")

## 8. Matrices, plots, actual-answer reports and human labels

Human semantic categories stay blank until reviewed. The exports do not infer
fabrication/refusal from ROUGE. Attack/durability decision fields stay null.

In [ ]:
def export_seed_report(folder, stages, config, author_blocks):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    baseline = stages["retain95_bf16"]
    overview, components = [], []
    for name, frame in stages.items():
        summary, parts = summarize(frame, baseline)
        overview.append({"stage": name, **summary})
        parts.insert(0, "stage", name)
        components.append(parts)
    matrix = pd.DataFrame(overview)
    matrix.to_csv(folder / "comparison_matrix.csv", index=False)
    pd.concat(components).to_csv(folder / "utility_components.csv", index=False)
    fq, directional = quantization_comparisons(
        stages, config.bootstrap_repeats, author_blocks=author_blocks
    )
    fq.to_csv(folder / "forget_quality_fixed_and_matched.csv", index=False)
    directional.to_csv(folder / "paired_directional_recovery.csv", index=False)
    frames = []
    for name, frame in stages.items():
        frame = frame.copy()
        frame["stage"] = name
        frames.append(frame)
    long = pd.concat(frames, ignore_index=True)
    long.to_csv(folder / "all_per_question.csv", index=False)
    answers = long.pivot(
        index=["split", "id", "question", "ground_truth"],
        columns="stage",
        values="generated_answer",
    ).reset_index()
    answers.to_csv(folder / "answers_side_by_side.csv", index=False)
    style = "<meta charset='utf-8'><style>body{font:14px system-ui;margin:24px}table{border-collapse:collapse}td,th{border:1px solid #aaa;padding:8px;vertical-align:top;white-space:pre-wrap;min-width:150px}th{background:#eef;position:sticky;top:0}</style>"
    (folder / "answers_side_by_side.html").write_text(
        style + answers.to_html(index=False, escape=True), encoding="utf-8"
    )
    if "npo_adapter" in stages:
        before = stages["npo_adapter"].set_index("id").sort_index()
        after = stages["npo_bf16"].set_index("id").sort_index()
        if not before.index.equals(after.index):
            raise ValueError("Adapter/merged evaluation rows differ")
        merge = []
        for split in before.split.unique():
            ids = before.index[before.split == split]
            for metric in (
                "answer_probability",
                "answer_nll",
                "rouge_l_recall",
                "source_geometric_truth_ratio_log",
            ):
                merge.append(
                    {
                        "split": split,
                        "metric": metric,
                        "merged_minus_adapter": float(
                            (after.loc[ids, metric] - before.loc[ids, metric]).mean()
                        ),
                    }
                )
        pd.DataFrame(merge).to_csv(folder / "merge_drift.csv", index=False)
    # Labels are deliberately empty. A human must make semantic judgments.
    annotations = long[
        [
            "stage",
            "split",
            "id",
            "question",
            "ground_truth",
            "generated_answer",
            "generation_hit_limit",
        ]
    ].copy()
    annotations["category"] = ""
    annotations["reviewer"] = ""
    annotations["notes"] = ""
    annotation_path = folder / "human_audit_template.csv"
    if not annotation_path.exists():
        annotations.to_csv(annotation_path, index=False)
    status = {
        "core_evaluation": "measured",
        "geometric_FQ_fixed_and_matched": "measured",
        "arithmetic_model_utility": "measured",
        "directional_truth_ratio_bootstrap": "measured",
        "KL_diagnostic": "measured" if config.evaluate_kl else "not requested",
        "quantization_modes_requested": config.quantization_modes,
        "human_semantic_audit": "awaiting manual labels; no rates inferred from ROUGE",
        "membership_inference": "not measured; answer-token/min-k scores alone are not an MIA without matched nonmembers",
        "FADE": "not measured",
        "second_quantizer_family": "not measured",
        "relearning": "not measured",
        "weight_quantization_bin_analysis": "not measured",
        "attack_detected": None,
        "durability_pass": None,
    }
    write_json(folder / "evaluation_coverage.json", status)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    indexed = matrix.set_index("stage")
    for ax, column in zip(
        axes, ("forget05_answer_probability", "forget05_rouge_l_recall", "model_utility")
    ):
        indexed[column].plot.bar(ax=ax)
        ax.set_title(column)
        ax.set_xlabel("")
        ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    fig.savefig(folder / "comparison.png", dpi=160)
    plt.close(fig)
    report = style + "<h1>LoRA NPO+RT / merged-model quantization</h1>"
    report += "<p>Source-style sampled decoding is primary. Inspect adapter-to-merge drift before quantization changes. KS non-rejection does not prove erasure.</p>"
    for title, frame in (
        ("Stage metrics", matrix),
        ("Fixed and matched geometric FQ", fq),
        ("Paired quantization changes", directional),
    ):
        report += f"<h2>{html.escape(title)}</h2>" + frame.round(6).to_html(
            index=False, escape=True
        )
    report += "<p><a href='answers_side_by_side.html'>All model-generated answers</a></p><img src='comparison.png' style='max-width:100%'>"
    report += "<h2>Study coverage</h2>" + pd.Series(status).to_frame("status").to_html(escape=True)
    (folder / "report.html").write_text(report, encoding="utf-8")
    return matrix


def export_cross_seed(root, matrices, expected_seeds):
    frames = []
    for seed, frame in matrices.items():
        frame = frame.copy()
        frame.insert(0, "seed", seed)
        frames.append(frame)
    values = pd.concat(frames, ignore_index=True)
    values.to_csv(Path(root) / "all_seeds_metrics.csv", index=False)
    numeric = [c for c in values.select_dtypes("number").columns if c != "seed"]
    aggregate = values.groupby("stage")[numeric].agg(["mean", "std", "count"])
    aggregate.to_csv(Path(root) / "cross_seed_mean_std.csv")
    write_json(
        Path(root) / "seed_coverage.json",
        {
            "expected_seeds": expected_seeds,
            "completed_seeds": list(matrices),
            "complete": set(matrices) == set(expected_seeds),
            "minimum_three_seeds_met": len(matrices) >= 3,
            "interpretation": "Seed means/spread are descriptive; fewer than three seeds is a pilot and bootstrap rows are not independent training runs",
        },
    )
    return aggregate


register_notebook_cell("reporting")

In [ ]:
CATEGORIES = {"correct", "partial", "refusal", "gibberish", "fabrication", "substitution", "other"}


def summarize_labels(path, output):
    data = pd.read_csv(path).fillna("")
    for key in ("stage", "split", "id", "category", "reviewer"):
        if key not in data:
            raise ValueError(f"Missing annotation field: {key}")
    labeled = data.loc[data.category.str.strip().ne("")].copy()
    if not set(labeled.category).issubset(CATEGORIES):
        raise ValueError("Unknown category; follow the supplied human audit rubric")
    if (labeled.reviewer.str.strip() == "").any():
        raise ValueError("Every judgment must identify its reviewer")
    totals = data.groupby(["stage", "split"]).size().rename("available")
    counts = labeled.groupby(["stage", "split", "category"]).size().rename("count").reset_index()
    reviewed = labeled.groupby(["stage", "split"]).size().rename("reviewed")
    counts = counts.merge(totals.reset_index(), on=["stage", "split"]).merge(
        reviewed.reset_index(), on=["stage", "split"]
    )
    counts["rate_among_reviewed"] = counts["count"] / counts["reviewed"]
    counts["review_coverage"] = counts["reviewed"] / counts["available"]
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    counts.to_csv(output, index=False)
    return counts


register_notebook_cell("human-labels")

## 9. Full experiment and merge/quantization order

Six full/retain95 precision controls are shared across training seeds. For each
seed: periodic complete TOFU evaluations → final unmerged adapter → FP32 merge
probes → BF16 save/reload → independent INT8 and NF4 → seed report. The final
checkpoint is deployed; no best-Forget05 checkpoint is selected.

In [ ]:
def environment(config):
    names = [
        "torch",
        "transformers",
        "peft",
        "accelerate",
        "numpy",
        "pandas",
        "scipy",
        "rouge-score",
        "safetensors",
    ]
    if config.quantization_modes:
        names.append("bitsandbytes")
    versions = {name: importlib.metadata.version(name) for name in names}
    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "packages": versions,
        "cuda": torch.version.cuda,
        "ram_available_bytes": psutil.virtual_memory().available,
        "requested_device": config.device,
        "dtype": config.dtype,
    }
    if str(config.device).startswith("cuda"):
        if not torch.cuda.is_available():
            raise ValueError("CUDA-enabled PyTorch and the correct notebook kernel are required")
        if (
            config.dtype != "bfloat16"
            or not torch.cuda.is_bf16_supported()
            or torch.cuda.get_device_capability(config.device)[0] < 8
        ):
            raise ValueError("Native BF16 is required for this declared local configuration")
        free, total = torch.cuda.mem_get_info(config.device)
        if free < 7 * 1024**3:
            raise ValueError(
                "Close other GPU applications; at least 7 GiB free is required to start"
            )
        info.update(
            gpu=torch.cuda.get_device_name(config.device),
            free_vram_bytes=free,
            total_vram_bytes=total,
        )
        existing = Path(config.output_root).resolve()
        while not existing.exists():
            existing = existing.parent
        disk_free = shutil.disk_usage(existing).free
        if disk_free < 12 * 1024**3:
            raise ValueError(
                "Reserve at least 12 GiB on the output SSD for three merged 1B checkpoints and adapter/results state"
            )
        if info["ram_available_bytes"] < 2 * 1024**3:
            raise ValueError(
                "Close other applications; at least 2 GiB free system RAM is required to start"
            )
        info["output_disk_free_bytes"] = disk_free
    torch.set_num_threads(min(8, __import__("os").cpu_count() or 1))
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
    return info


def quantization_preflight(config):
    if not config.quantization_modes:
        return
    import bitsandbytes as bnb

    dtype = getattr(torch, config.dtype)
    with torch.no_grad():
        for mode in config.quantization_modes:
            layer = (
                bnb.nn.Linear8bitLt(32, 32, bias=False, has_fp16_weights=False)
                if mode == "int8"
                else bnb.nn.Linear4bit(
                    32,
                    32,
                    bias=False,
                    compute_dtype=dtype,
                    quant_type="nf4",
                    compress_statistics=True,
                )
            )
            layer.requires_grad_(False)
            layer = layer.to(device=config.device, dtype=dtype)
            output = layer(torch.randn(1, 32, device=config.device, dtype=dtype))
            if not torch.isfinite(output).all():
                raise ValueError(f"Broken {mode} backend")
            del layer, output
    release_memory()


def merge_export(model, tokenizer, forget, config, folder):
    """Merge in FP32 on GPU, avoiding a full FP32 CPU copy, then deploy BF16."""
    folder = Path(folder)
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.to(dtype=torch.float32)
    probes, before = forget[:3], []
    with torch.no_grad():
        for row in probes:
            inputs, _ = one_batch(row, config.device, tokenizer.pad_token_id)
            before.append(model(**inputs, use_cache=False).logits[:, -3:].float().cpu())
        merged = model.merge_and_unload(safe_merge=True)
        errors = []
        for row, expected in zip(probes, before):
            inputs, _ = one_batch(row, config.device, tokenizer.pad_token_id)
            actual = merged(**inputs, use_cache=False).logits[:, -3:].float().cpu()
            errors.append(float((actual - expected).abs().max()))
            torch.testing.assert_close(actual, expected, atol=3e-4, rtol=1e-4)
    merged.to(dtype=getattr(torch, config.dtype))
    merged.to("cpu")
    merged.save_pretrained(folder, safe_serialization=True, max_shard_size="1GB")
    tokenizer.save_pretrained(folder)
    receipt = {
        "status": "complete",
        "merge_precision": "float32",
        "deployment_dtype": config.dtype,
        "fp32_merge_probe_max_abs_errors": errors,
        "probe_positions": "last three logits including scored answer/EOS positions",
        "quantized_during_training": False,
        "weight_hashes": {p.name: sha256(p) for p in folder.glob("*.safetensors")},
    }
    write_json(folder / "merge_complete.json", receipt)
    return merged, receipt


def run(config, resume_dir=None):
    config.validate()
    env = environment(config)
    quantization_preflight(config)
    paths, tokenizer, data, data_receipts = resolve_inputs(config)
    forget, retain, packs, selection = prepare_examples(data, tokenizer, config)
    assert_notebook_ready()
    code = dict(NOTEBOOK_CODE)
    identity = digest(
        {
            "config": config.to_dict(),
            "inputs": data_receipts,
            "selection": selection,
            "code": code,
            "packages": env["packages"],
        }
    )
    if resume_dir:
        root = Path(resume_dir).resolve()
        manifest = json.loads((root / "manifest.json").read_text())
        if manifest["identity"] != identity:
            raise ValueError("Resume config/input/code/package identity changed; create a new run")
    else:
        root = Path(config.output_root) / (
            time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns() % 1_000_000:06d}"
        )
        root.mkdir(parents=True, exist_ok=False)
        write_json(
            root / "manifest.json",
            {
                "identity": identity,
                "config": config.to_dict(),
                "environment": env,
                "code": code,
                "inputs": data_receipts,
                "models": {key: str(value) for key, value in paths.items()},
                "model_sha256": {
                    "full": PINS["target"]["checkpoint_sha256"],
                    "retain95": PINS["retain_only_oracle"]["checkpoint_sha256"],
                },
            },
        )
        write_json(root / "evaluation_selection.json", selection)
    print(f"Run directory: {root}", flush=True)
    notify_notebook("run_start", root=root, config=config)
    profiles = ("source_sampled", "legacy_greedy96", "long_greedy")
    controls, matrices = {}, {}

    @preserve_rng()
    def measure(
        stage,
        path,
        folder,
        oracle=None,
        quant=None,
        loaded=None,
        extra=None,
        selected_profiles=profiles,
    ):
        model, reference = loaded, None
        owned = loaded is None
        try:
            if owned:
                model = model_loader(path, config, quant)
            details = model_metadata(model, quant)
            if config.evaluate_kl:
                reference = (
                    model
                    if path == paths["full"] and quant is None and owned
                    else model_loader(paths["full"], config)
                )
            binding = {
                "run_identity": identity,
                "stage": stage,
                "model_path": str(path),
                "model": details,
                "KL_reference": PINS["target"]["checkpoint_sha256"] if config.evaluate_kl else None,
                "KL_positions": "all 200 source positions including prompt/padding",
                **(extra or {}),
            }
            frame = evaluate(
                model,
                tokenizer,
                packs,
                config,
                folder,
                binding,
                reference=reference,
                oracle=oracle,
                profiles=selected_profiles,
            )
            write_json(Path(folder) / "model_identity.json", binding)
            stage_summary, _ = summarize(frame, oracle)
            notify_notebook(
                "evaluation_complete", stage=stage, folder=Path(folder), summary=stage_summary
            )
            return frame
        finally:
            reference = None
            if owned:
                model = None
            release_memory()

    try:
        controls["full_bf16"] = measure("full_bf16", paths["full"], root / "controls/full_bf16")
        controls["retain95_bf16"] = measure(
            "retain95_bf16", paths["retain95"], root / "controls/retain95_bf16"
        )
        if config.quantized_controls:
            for mode in config.quantization_modes:
                for kind in ("full", "retain95"):
                    name = f"{kind}_{mode}"
                    controls[name] = measure(
                        name,
                        paths[kind],
                        root / "controls" / name,
                        oracle=controls["retain95_bf16"],
                        quant=mode,
                    )
        author_blocks = {p["question"]: p["author_block"] for p in packs["forget05"]}
        for seed in config.seeds:
            seed_folder = root / f"seed_{seed}"
            complete = seed_folder / "seed_complete.json"
            if complete.is_file():
                old = json.loads(complete.read_text())
                for name, expected in old["report_hashes"].items():
                    if sha256(seed_folder / name) != expected:
                        raise ValueError("Completed seed report was modified")
                matrices[seed] = pd.read_csv(seed_folder / "comparison_matrix.csv")
                continue
            seed_folder.mkdir(exist_ok=True)
            base = model_loader(paths["full"], config)
            model = attach_lora(base, config, seed)
            del base

            def during_training(current, step, training_identity):
                total = len(forget) * config.nominal_epochs // config.effective_batch
                if step == total:
                    return  # Final evaluation below includes all three profiles on the final checkpoint.
                location = seed_folder / "epoch_evaluations" / f"update_{step:06d}"
                frame = measure(
                    f"npo_adapter_step_{step}",
                    seed_folder / "adapter",
                    location,
                    oracle=controls["retain95_bf16"],
                    loaded=current,
                    extra={"training_identity": training_identity, "update": step},
                    selected_profiles=("source_sampled",),
                )
                row, _ = summarize(frame, controls["retain95_bf16"])
                write_json(
                    location / "epoch_summary.json",
                    {
                        "seed": seed,
                        "update": step,
                        "effective_forget_passes": step * config.effective_batch / len(forget),
                        **row,
                    },
                )
                milestone = location / "adapter"
                milestone_receipt = location / "adapter_checkpoint.json"
                if milestone_receipt.exists():
                    archived = json.loads(milestone_receipt.read_text())
                    for name, expected in archived["hashes"].items():
                        if sha256(milestone / name) != expected:
                            raise ValueError("Archived epoch adapter was modified")
                else:
                    current.save_pretrained(milestone, safe_serialization=True)
                    write_json(
                        milestone_receipt,
                        {
                            "update": step,
                            "training_identity": training_identity,
                            "hashes": {
                                p.name: sha256(p) for p in milestone.glob("*") if p.is_file()
                            },
                        },
                    )

            training = train(
                model,
                tokenizer,
                forget,
                retain,
                config,
                seed,
                seed_folder,
                {"run_identity": identity, "seed": seed},
                during_training,
                resume=bool(resume_dir and list((seed_folder / "checkpoints").glob("step-*"))),
            )
            stages = dict(controls)
            stages["npo_adapter"] = measure(
                "npo_adapter",
                seed_folder / "adapter",
                seed_folder / "npo_adapter",
                oracle=controls["retain95_bf16"],
                loaded=model,
                extra={"training_identity": training["identity"]},
            )
            merged_path = seed_folder / "merged_model"
            merged, merge = merge_export(model, tokenizer, forget, config, merged_path)
            del model, merged
            release_memory()
            stages["npo_bf16"] = measure(
                "npo_bf16",
                merged_path,
                seed_folder / "npo_bf16",
                oracle=controls["retain95_bf16"],
                extra={"merge": merge},
            )
            for mode in config.quantization_modes:
                stages[f"npo_{mode}"] = measure(
                    f"npo_{mode}",
                    merged_path,
                    seed_folder / f"npo_{mode}",
                    oracle=controls["retain95_bf16"],
                    quant=mode,
                    extra={"merge": merge},
                )
            matrices[seed] = export_seed_report(seed_folder, stages, config, author_blocks)
            notify_notebook("seed_complete", seed=seed, folder=seed_folder, matrix=matrices[seed])
            epoch_rows = [
                json.loads(p.read_text())
                for p in sorted((seed_folder / "epoch_evaluations").glob("*/epoch_summary.json"))
            ]
            final_summary, _ = summarize(stages["npo_adapter"], controls["retain95_bf16"])
            epoch_rows.append(
                {
                    "seed": seed,
                    "update": training["updates"],
                    "effective_forget_passes": training["updates"]
                    * config.effective_batch
                    / len(forget),
                    **final_summary,
                }
            )
            pd.DataFrame(epoch_rows).to_csv(seed_folder / "epoch_metrics.csv", index=False)
            reports = [
                "comparison_matrix.csv",
                "utility_components.csv",
                "forget_quality_fixed_and_matched.csv",
                "paired_directional_recovery.csv",
                "merge_drift.csv",
                "answers_side_by_side.csv",
                "epoch_metrics.csv",
            ]
            reports += [
                str(p.relative_to(seed_folder))
                for p in seed_folder.glob("npo*/**/*")
                if p.is_file()
                and p.name in ("per_question.csv", "per_question.jsonl", "metadata.json")
            ]
            write_json(
                complete,
                {
                    "status": "complete",
                    "seed": seed,
                    "training_identity": training["identity"],
                    "report_hashes": {name: sha256(seed_folder / name) for name in reports},
                },
            )
            export_cross_seed(root, matrices, config.seeds)
        aggregate = export_cross_seed(root, matrices, config.seeds)
        write_json(
            root / "run_complete.json",
            {
                "status": "complete",
                "identity": identity,
                "seeds": config.seeds,
                "quantization_modes": config.quantization_modes,
                "minimum_three_seeds_met": len(matrices) >= 3,
                "attack_detected": None,
                "durability_pass": None,
            },
        )
        print(aggregate.to_string())
        print(f"All measured results: {root}")
        return root
    except BaseException as exc:
        write_json(
            root / "run_failure.json",
            {
                "error": repr(exc),
                "identity": identity,
                "completed_seeds": list(matrices),
                "instruction": "Use --resume with this run directory and unchanged config/code.",
            },
        )
        raise


register_notebook_cell("pipeline")

## 10. Optional run-2 reanalysis definition

This helper uses only saved likelihoods and verifies the original recorded
hashes. It is included here so reanalysis also requires no local Python files.
It does not modify the original second run or perform model inference.

In [ ]:
def reanalyse_second_run(run_dir, output_dir, repeats=2000):
    run_dir, output_dir = Path(run_dir).resolve(), Path(output_dir).resolve()
    if output_dir == run_dir or run_dir in output_dir.parents:
        raise ValueError("Derived analysis must live outside the immutable original run")
    output_dir.mkdir(parents=True, exist_ok=True)
    stages, inventory = load_existing_stages(run_dir)
    if not all(
        s in stages
        for s in (
            "npo_bf16",
            "retain95_bf16",
            "npo_int8",
            "npo_nf4",
            "retain95_int8",
            "retain95_nf4",
        )
    ):
        raise ValueError("All six candidate/oracle per-question CSVs required")
    # Verify contiguous source order against the saved question manifest before
    # using TOFU's 20-questions-per-author block bootstrap.
    questions_path = run_dir / "evaluation_questions.json"
    blocks = None
    if questions_path.is_file():
        questions = json.loads(questions_path.read_text(encoding="utf-8"))["forget05"]
        before = stages["npo_bf16"].query("split == 'forget05'").sort_values("id")
        if len(questions) == 200 and before.question.tolist() == [r["question"] for r in questions]:
            blocks = {row["question"]: i // 20 for i, row in enumerate(questions)}
    fq, paired = quantization_comparisons(stages, repeats=repeats, author_blocks=blocks)
    fq.to_csv(output_dir / "reanalysis_forget_quality.csv", index=False)
    paired.to_csv(output_dir / "reanalysis_directional_qra.csv", index=False)
    overview, components = [], []
    for name, frame in stages.items():
        summary, parts = summarize(frame, stages["retain95_bf16"])
        overview.append({"stage": name, **summary})
        parts.insert(0, "stage", name)
        components.append(parts)
    pd.DataFrame(overview).to_csv(output_dir / "reanalysis_stage_metrics.csv", index=False)
    pd.concat(components).to_csv(output_dir / "reanalysis_utility_components.csv", index=False)
    termination_rows = []
    for name, frame in stages.items():
        for split, group in frame.groupby("split"):
            sequences = [
                ast.literal_eval(x) if isinstance(x, str) else x for x in group.generated_token_ids
            ]
            termination_rows.append(
                {
                    "stage": name,
                    "split": split,
                    "n": len(group),
                    "mean_generated_tokens": sum(map(len, sequences)) / len(sequences),
                    "legacy_96_token_cap_rate": float(group.generation_hit_limit.mean()),
                    "fraction_containing_source_stop": sum(
                        any(t in (128001, 128008, 128009) for t in seq) for seq in sequences
                    )
                    / len(sequences),
                    "fraction_containing_other_source_stop": sum(
                        any(t in (128001, 128008) for t in seq) for seq in sequences
                    )
                    / len(sequences),
                }
            )
    pd.DataFrame(termination_rows).to_csv(
        output_dir / "generation_termination_audit.csv", index=False
    )
    history = pd.read_csv(run_dir / "training_log.csv")
    plan = json.loads((run_dir / "training_plan.json").read_text())
    receipt = json.loads((run_dir / "npo_model/training_complete.json").read_text())
    unique_retain = len(set(plan["retain_pairs"]))
    verification = {
        "status": "derived-analysis-complete",
        "source_run": str(run_dir),
        "source_run_id": run_dir.name,
        "source_files": inventory,
        "reanalysis_code_sha256": implementation_hash("run2-reanalysis"),
        "metrics_code_sha256": implementation_hash("metrics"),
        "analysis_environment": {
            "python": platform.python_version(),
            "packages": {
                name: importlib.metadata.version(name) for name in ("numpy", "pandas", "scipy")
            },
        },
        "bootstrap_repeats": repeats,
        "question_bootstrap": "exploratory due to within-author dependence",
        "author_block_bootstrap": "10 contiguous pinned TOFU author blocks"
        if blocks
        else "not available",
        "completed_updates": int(receipt["updates"]),
        "training_log_rows": len(history),
        "unique_retain_rows": unique_retain,
        "available_retain_rows": 3800,
        "training_seconds_from_logged_rows": float(history.seconds.sum()),
        "training_log_timing_limit": "Resume history may omit post-commit optimizer timing fields; not a complete wall-clock estimate",
        "old_interruption_record_preserved": (run_dir / "training_failure.json").is_file(),
        "model_weight_files_available": bool(list((run_dir / "npo_model").glob("*.safetensors"))),
        "model_weight_integrity": "not reverified: supplied archive has receipts, not checkpoint tensors",
        "attack_detected": None,
        "durability_pass": None,
        "missing_study_components": [
            "membership inference with matched nonmembers",
            "FADE",
            "human-coded semantic audit",
            "additional quantizers",
            "relearning",
            "weight/bin analysis",
            "multiple training seeds",
        ],
    }
    write_json(output_dir / "VERIFICATION.json", verification)
    output_hashes = {
        p.name: sha256(p)
        for p in output_dir.iterdir()
        if p.is_file() and p.name != "DERIVED_MANIFEST.json"
    }
    write_json(
        output_dir / "DERIVED_MANIFEST.json",
        {"inputs_sha256": digest(inventory), "outputs": output_hashes},
    )
    print(fq.to_string(index=False))
    print(paired.loc[paired.metric == "source_geometric_truth_ratio_log"].to_string(index=False))
    return verification


register_notebook_cell("run2-reanalysis")

## 11. Engineering tests — executed inside the notebook

Small synthetic Llama fixtures test the **functions defined above**, not a
separate implementation. They require no downloaded model/data and do not
start the 1B experiment. Coverage: objective/gradient/masking parity, reference
freezing, sampling, logical microbatch weighting, exact resume, corruption
fallback, merge/reload, all evaluation exports and cache tampering. A CUDA-only
test additionally exercises actual LoRA backward and INT8/NF4 conversion.

Each test has its own fixture and output folder. All tests are shown below;
failures block the full run. A CUDA skip is explicitly reported as a skip.

In [ ]:
def make_tiny_fixture(tmp_path):
    torch.set_num_threads(1)
    vocab = {
        t: i
        for i, t in enumerate(
            [
                "<pad>",
                "<eot>",
                "<unk>",
                "<system>",
                "<user>",
                "<assistant>",
                "You",
                "are",
                "a",
                "helpful",
                "assistant",
                ".",
                "?",
                "A",
                "B",
                "C",
                "D",
            ]
            + [f"q{i}" for i in range(64)]
        )
    }
    backend = Tokenizer(WordLevel(vocab, unk_token="<unk>"))
    backend.pre_tokenizer = Whitespace()
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=backend,
        pad_token="<pad>",
        eos_token="<eot>",
        unk_token="<unk>",
        additional_special_tokens=["<system>", "<user>", "<assistant>"],
    )
    tokenizer.chat_template = "{% for m in messages %}{{ '<' + m['role'] + '>' }} {{ m['content'] }} <eot> {% endfor %}{% if add_generation_prompt %}<assistant> {% endif %}"
    cfg = LlamaConfig(
        vocab_size=len(vocab),
        hidden_size=16,
        intermediate_size=24,
        num_hidden_layers=1,
        num_attention_heads=2,
        num_key_value_heads=1,
        max_position_embeddings=128,
        tie_word_embeddings=True,
        attention_dropout=0.0,
        eos_token_id=1,
        pad_token_id=0,
    )
    torch.manual_seed(123)
    state = LlamaForCausalLM(cfg).state_dict()

    def make_model():
        model = LlamaForCausalLM(cfg)
        model.load_state_dict(state)
        return model

    config = Config(
        device="cpu",
        dtype="float32",
        seeds=[42],
        nominal_epochs=2,
        logical_micro_batch=2,
        device_micro_batch=2,
        accumulation=2,
        warmup_steps=1,
        evaluation_every_updates=2,
        training_max_length=64,
        evaluation_max_length=24,
        evaluation_retain_count=3,
        generation_batch_size=2,
        legacy_greedy_tokens=4,
        long_audit_tokens=8,
        long_audit_per_split=2,
        lora_rank=2,
        lora_alpha=4,
        gradient_checkpointing=True,
        quantization_modes=[],
        bootstrap_repeats=100,
        output_root=str(tmp_path / "runs"),
    )

    def row(i):
        return {
            "question": f"q{i} ?",
            "answer": "A",
            "paraphrased_answer": "B",
            "perturbed_answer": ["C", "D"],
        }

    data = {
        "forget05": [row(i) for i in range(8)],
        "retain95": [row(i) for i in range(8, 56)],
        "forget05_perturbed": [row(i) for i in range(8)],
        "retain_perturbed": [row(i) for i in range(8, 12)],
        "real_authors_perturbed": [row(i) for i in range(56, 58)],
        "world_facts_perturbed": [row(i) for i in range(58, 60)],
    }
    return (config, tokenizer, data, make_model)


register_notebook_cell("tiny-fixture")

In [ ]:
def test_declared_defaults_match_project_lora_policy():
    policy = PROJECT_LORA_POLICY
    cfg = Config()
    assert policy["decision"] == "DL-024"
    assert (cfg.lora_rank, cfg.lora_alpha, cfg.lora_dropout) == (
        policy["r"],
        policy["lora_alpha"],
        policy["lora_dropout"],
    )
    assert cfg.lora_targets == policy["target_modules"]


def test_sequence_formula_gradients_masks_and_reference():
    torch.manual_seed(4)
    logits = torch.randn(2, 6, 11, dtype=torch.float64, requires_grad=True)
    reference = torch.randn_like(logits, requires_grad=True)
    labels = torch.tensor([[-100, -100, 2, 3, 4, -100], [-100, 2, 3, 4, 5, 6]])
    current, frozen = sequence_nll(logits, labels), sequence_nll(reference, labels)
    oracle = torch.stack(
        [
            sum(
                -logits[i, t - 1].log_softmax(-1)[labels[i, t]]
                for t in range(1, 6)
                if labels[i, t] != -100
            )
            for i in range(2)
        ]
    )
    loss = npo_loss(current, frozen)
    expected = -20 * functional.logsigmoid(0.1 * (oracle - frozen.detach())).mean()
    torch.testing.assert_close(current, oracle)
    torch.testing.assert_close(loss, expected)
    torch.testing.assert_close(
        torch.autograd.grad(loss, logits, retain_graph=True)[0],
        torch.autograd.grad(expected, logits, retain_graph=True)[0],
    )
    loss.backward()
    assert reference.grad is None and not logits.grad[0, 0].any() and not logits.grad[:, -1].any()
    padded = functional.pad(logits.detach(), (0, 0, 0, 4))
    torch.testing.assert_close(
        sequence_nll(padded, functional.pad(labels, (0, 4), value=-100)), current.detach()
    )
    initial = torch.tensor([2.0, 5.0], requires_grad=True)
    value = npo_loss(initial, initial.detach())
    assert float(value) == pytest.approx(20 * math.log(2))
    value.backward()
    assert (initial.grad < 0).all()
    with pytest.raises(ValueError):
        npo_loss(initial, initial, beta=0)


def test_fresh_sampler_covers_broad_retain_and_is_replayable():
    cfg = Config()
    plan = build_plan(200, 3800, cfg, 42)
    assert plan == build_plan(200, 3800, cfg, 42)
    assert plan["steps"] == 62 and plan["effective_data_passes"] == 9.92
    assert plan["unique_retain_rows"] == 1984
    assert plan["retain_batches"] != build_plan(200, 3800, cfg, 43)["retain_batches"]
    cfg.retain_sampling = "source_fixed"
    source = build_plan(200, 3800, cfg, 42)
    assert source["unique_retain_rows"] == 195
    expected = [
        (i + torch.randint(3800, (1,), generator=torch.Generator().manual_seed(i)).item()) % 3800
        for i in range(200)
    ]
    for f, r in zip(source["forget_batches"], source["retain_batches"]):
        assert r == [expected[i] for i in f]


def test_masking_selection_and_lora_reference(tiny):
    cfg, tokenizer, data, make_model = tiny
    forget, retain, packs, selection = prepare_examples(data, tokenizer, cfg)
    assert selection == prepare_examples(data, tokenizer, cfg)[3]
    assert not {p["question"] for p in packs["retain"]} & {r["question"] for r in data["forget05"]}
    for row in forget:
        assert all(t == -100 for t in row["labels"][: row["prompt_length"]])
        assert row["labels"][-1] == tokenizer.eos_token_id
    with pytest.raises(ValueError, match="truncation"):
        encode_qa(tokenizer, "q0 ?", "A", cfg, max_length=2)
    base = make_model().eval()
    expected = forward_sequence(base, forget[:2], tokenizer, cfg)[0].detach()
    model = attach_lora(base, cfg, 42)
    original = tensor_fingerprint(model, skip_adapters=True)
    assert all("lora_" in name for name, p in model.named_parameters() if p.requires_grad)
    torch.testing.assert_close(frozen_reference_nll(model, forget[:2], tokenizer, cfg), expected)
    for name, parameter in model.named_parameters():
        if "lora_B" in name:
            with torch.no_grad():
                parameter.add_(0.5)
    torch.testing.assert_close(frozen_reference_nll(model, forget[:2], tokenizer, cfg), expected)
    assert tensor_fingerprint(model, skip_adapters=True) == original


def test_device_microbatch_preserves_logical_retain_weighting(tiny):
    cfg, tokenizer, data, make_model = tiny
    forget, retain, _, _ = prepare_examples(data, tokenizer, cfg)
    retain[1] = encode_qa(tokenizer, "q9 ?", "A B C D", cfg)
    a = attach_lora(make_model(), cfg, 42)
    b = copy.deepcopy(a)
    current, _ = forward_sequence(a, forget[:2], tokenizer, cfg)
    reference = frozen_reference_nll(a, forget[:2], tokenizer, cfg)
    retained, count = forward_sequence(a, retain[:2], tokenizer, cfg)
    (npo_loss(current, reference) + retained.sum() / count).backward()
    for f, r in zip(forget[:2], retain[:2]):
        z, _ = forward_sequence(b, [f], tokenizer, cfg)
        ref = frozen_reference_nll(b, [f], tokenizer, cfg)
        (npo_loss(z, ref) / 2).backward()
        z, _ = forward_sequence(b, [r], tokenizer, cfg)
        (z.sum() / count).backward()
    for (_, pa), (_, pb) in zip(a.named_parameters(), b.named_parameters()):
        if pa.requires_grad:
            torch.testing.assert_close(pa.grad, pb.grad, atol=2e-6, rtol=2e-4)


def sample_frame(shift=0.0):
    rows = []
    for split in ("forget05", "retain", "real_authors", "world_facts"):
        for i in range(5):
            rows.append(
                {
                    "id": f"{split}:{i}",
                    "split": split,
                    "question": f"q{split}{i}",
                    "ground_truth": "A",
                    "answer_nll": 0.3 + i * 0.02,
                    "paraphrase_mean_nll": 0.5 + i * 0.1,
                    "perturbed_mean_nll": [1.0 + i * 0.2 + shift, 3.0 + i * 0.3 + shift],
                    "rouge_l_recall": 0.5,
                    "token_f1": 0.4,
                    "exact_match": 0.0,
                }
            )
    return enrich(pd.DataFrame(rows))


def test_arithmetic_utility_and_geometric_fq_are_distinct():
    values = truth_logs(0.5, [1.0, 3.0])
    assert values["source_geometric_truth_ratio_log"] == 1.5
    assert math.exp(values["arithmetic_truth_ratio_log"]) == pytest.approx(
        (np.exp(-1) + np.exp(-3)) / 2 / np.exp(-0.5)
    )
    frame = sample_frame()
    summary, components = summarize(frame, frame)
    assert summary["forget_quality"] == 1.0
    assert summary["ks_statistic"] == 0.0
    vector = (
        components[["utility_probability", "rouge_l_recall", "arithmetic_truth_score"]]
        .to_numpy()
        .flatten()
    )
    assert len(vector) == 9
    assert summary["model_utility"] == pytest.approx(len(vector) / np.reciprocal(vector).sum())


def test_matched_precision_uses_geometric_ratio_and_aligns_rows():
    before, after, oracle = sample_frame(), sample_frame(0.1), sample_frame(0.2)
    stages = {"npo_bf16": before, "npo_nf4": after, "retain95_bf16": before, "retain95_nf4": oracle}
    fq, changes = quantization_comparisons(stages, repeats=100)
    got = fq.query("stage == 'npo_nf4' and oracle_policy == 'matched_precision'").iloc[0]
    expected = ks_2samp(
        after.query("split == 'forget05'").source_geometric_truth_ratio_log,
        oracle.query("split == 'forget05'").source_geometric_truth_ratio_log,
    )
    assert got.ks_statistic == expected.statistic and got.forget_quality == expected.pvalue
    directional = changes.query("metric == 'source_geometric_truth_ratio_log'").iloc[0]
    assert directional.mean_change == pytest.approx(0.1)
    assert directional.ci_low == pytest.approx(0.1)
    assert directional.fraction_increased == 1.0
    # Equal geometric means but sharply different arithmetic means must give
    # geometric KS D=0 while the separately named arithmetic diagnostic differs.
    geometric_same = sample_frame()
    arithmetic_different = sample_frame()
    geometric_same["perturbed_mean_nll"] = [[0.1, 7.9] for _ in range(len(geometric_same))]
    arithmetic_different["perturbed_mean_nll"] = [
        [4.0, 4.0] for _ in range(len(arithmetic_different))
    ]
    distinct = forget_quality(enrich(geometric_same), enrich(arithmetic_different))
    assert distinct["ks_statistic"] == 0.0 and distinct["forget_quality"] == 1.0
    assert distinct["paper_arithmetic_ks_statistic_diagnostic"] == 1.0
    bad = after.copy()
    bad.loc[0, "question"] = "different question"
    with pytest.raises(ValueError, match="question"):
        align(bad, before)


def test_cluster_bootstrap_is_paired_and_reproducible():
    before = np.arange(6.0)
    after = before + np.array([0.1, 0.1, -0.2, -0.2, 0.3, 0.3])
    a = paired_bootstrap(before, after, repeats=200, clusters=[0, 0, 1, 1, 2, 2])
    b = paired_bootstrap(before, after, repeats=200, clusters=[0, 0, 1, 1, 2, 2])
    assert a == b and a["resampling_unit"] == "author-block"
    assert a["mean_change"] == pytest.approx(1 / 15)
    with pytest.raises(ValueError):
        paired_bootstrap([1], [float("nan")])
    clusters = np.repeat(np.arange(3), 200)
    repeated = np.repeat([0.1, -0.2, 0.3], 200)
    cluster_ci = paired_bootstrap(np.zeros(600), repeated, repeats=200, clusters=clusters)
    question_ci = paired_bootstrap(np.zeros(600), repeated, repeats=200)
    assert cluster_ci["ci_high"] - cluster_ci["ci_low"] > 5 * (
        question_ci["ci_high"] - question_ci["ci_low"]
    )


def test_resume_matches_uninterrupted_and_frozen_backbone(tiny, tmp_path):
    cfg, tokenizer, data, make_model = tiny
    forget, retain, _, _ = prepare_examples(data, tokenizer, cfg)
    full = attach_lora(make_model(), cfg, 42)
    train(full, tokenizer, forget, retain, cfg, 42, tmp_path / "full", {"fixture": True})
    interrupted = attach_lora(make_model(), cfg, 42)
    before = tensor_fingerprint(interrupted, skip_adapters=True)
    train(
        interrupted,
        tokenizer,
        forget,
        retain,
        cfg,
        42,
        tmp_path / "resumed",
        {"fixture": True},
        stop_after=2,
    )
    assert not (tmp_path / "resumed/training_complete.json").exists()
    resumed = attach_lora(make_model(), cfg, 42)
    receipt = train(
        resumed,
        tokenizer,
        forget,
        retain,
        cfg,
        42,
        tmp_path / "resumed",
        {"fixture": True},
        resume=True,
    )
    assert receipt["base_unchanged"]
    assert before == tensor_fingerprint(resumed, skip_adapters=True)
    for name, expected in full.state_dict().items():
        torch.testing.assert_close(resumed.state_dict()[name], expected, atol=0, rtol=0)
    assert len(checkpoint_candidates(tmp_path / "resumed/checkpoints")) == 2


def test_corrupt_checkpoint_falls_back_and_identity_is_checked(tiny, tmp_path):
    cfg, tokenizer, data, make_model = tiny
    forget, retain, _, _ = prepare_examples(data, tokenizer, cfg)
    model = attach_lora(make_model(), cfg, 42)
    result = train(
        model,
        tokenizer,
        forget,
        retain,
        cfg,
        42,
        tmp_path / "train",
        {"fixture": True},
        stop_after=2,
    )
    latest = checkpoint_candidates(tmp_path / "train/checkpoints")[0][1]
    (latest / "runtime.pt").write_bytes(b"corruption")
    fresh = attach_lora(make_model(), cfg, 42)
    optimizer = torch.optim.AdamW(
        [
            {
                "params": [p for p in fresh.parameters() if p.requires_grad],
                "weight_decay": cfg.weight_decay,
            },
            {"params": [], "weight_decay": 0.0},
        ],
        lr=cfg.learning_rate,
    )
    step, _, rejected = restore_checkpoint(
        tmp_path / "train/checkpoints", fresh, optimizer, result["identity"]
    )
    assert step == 1 and rejected
    with pytest.raises(ValueError, match="compatible"):
        restore_checkpoint(tmp_path / "train/checkpoints", fresh, optimizer, "different experiment")


def test_completed_pipeline_generates_all_outputs_and_reuses_evaluation(
    tiny, tmp_path, monkeypatch
):

    cfg, tokenizer, data, make_model = tiny
    full, retain = tmp_path / "full", tmp_path / "retain"
    make_model().save_pretrained(full, safe_serialization=True)
    make_model().save_pretrained(retain, safe_serialization=True)
    paths = {"full": full, "retain95": retain, "tokenizer": full}
    monkeypatch.setitem(
        globals(), "resolve_inputs", lambda _: (paths, tokenizer, data, {"fixture": True})
    )
    progress = NotebookProgress(2)
    monkeypatch.setitem(globals(), "_NOTEBOOK_OBSERVER", progress)
    root = run(cfg)
    assert len(progress.training) == 4 and len(progress.evaluations) >= 4
    assert json.loads((root / "run_complete.json").read_text())["status"] == "complete"
    seed = root / "seed_42"
    for name in (
        "comparison_matrix.csv",
        "forget_quality_fixed_and_matched.csv",
        "paired_directional_recovery.csv",
        "answers_side_by_side.html",
        "human_audit_template.csv",
        "epoch_metrics.csv",
        "merge_drift.csv",
    ):
        assert (seed / name).is_file()
    assert len(pd.read_csv(seed / "epoch_metrics.csv")) == 2
    primary = pd.read_csv(seed / "npo_bf16/per_question.csv")
    assert len(primary) == 15
    assert primary.eos_terminated.notna().all()
    assert (seed / "npo_bf16/legacy_greedy96/per_question.csv").is_file()
    assert (seed / "npo_bf16/long_greedy/per_question.csv").is_file()
    assert pd.read_csv(seed / "human_audit_template.csv").category.isna().all()
    immutable = sha256(seed / "npo_bf16/per_question.csv")
    run(cfg, resume_dir=root)
    assert sha256(seed / "npo_bf16/per_question.csv") == immutable
    # Tampering is not silently accepted as a cached completed result.
    with (seed / "npo_bf16/per_question.csv").open("a") as handle:
        handle.write("changed")
    with pytest.raises(ValueError, match="modified"):
        run(cfg, resume_dir=root)


@pytest.mark.skipif(
    not torch.cuda.is_available(), reason="Actual CUDA kernels require the target GPU"
)
def test_cuda_lora_backward_merge_and_real_quantization(tiny, tmp_path):
    pytest.importorskip("bitsandbytes")

    cfg, tokenizer, data, make_model = tiny
    cfg.device, cfg.dtype = "cuda:0", "bfloat16"
    cfg.quantization_modes = ["int8", "nf4"]
    quantization_preflight(cfg)
    forget, _, _, _ = prepare_examples(data, tokenizer, cfg)
    model = attach_lora(make_model().to(device=cfg.device, dtype=torch.bfloat16), cfg, 42)
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    current, _ = forward_sequence(model, forget[:2], tokenizer, cfg)
    frozen = frozen_reference_nll(model, forget[:2], tokenizer, cfg)
    npo_loss(current, frozen).backward()
    assert all(
        p.grad is not None and torch.isfinite(p.grad).all()
        for p in model.parameters()
        if p.requires_grad
    )
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4)
    optimizer.step()
    merged, _ = merge_export(model, tokenizer, forget, cfg, tmp_path / "merged")
    del merged, model, optimizer
    for mode in cfg.quantization_modes:
        quantized = model_loader(tmp_path / "merged", cfg, mode)
        values, _ = forward_sequence(quantized, forget[:1], tokenizer, cfg)
        assert torch.isfinite(values).all()
        del quantized


def test_notebook_identity_ignores_outputs_but_rejects_core_edits(tmp_path, monkeypatch):
    document = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    copy = tmp_path / "copied_notebook.ipynb"
    document["cells"][0]["metadata"]["test_output_metadata"] = "does not change core"
    copy.write_text(json.dumps(document), encoding="utf-8")
    monkeypatch.setitem(globals(), "NOTEBOOK_PATH", copy)
    assert_notebook_ready()
    cell = next(cell for cell in document["cells"] if cell["id"] == "objective")
    cell["source"] = "".join(cell["source"]) + "\n# changed core"
    copy.write_text(json.dumps(document), encoding="utf-8")
    with pytest.raises(ValueError, match="changed"):
        assert_notebook_ready()


def test_notebook_terminal_tee_preserves_console_and_file(tmp_path):
    console = io.StringIO()
    log = tmp_path / "session.txt"
    with contextlib.redirect_stdout(console), contextlib.redirect_stderr(console):
        with notebook_terminal_log(log):
            print("visible training update")
            sys.stderr.write("visible diagnostic\n")
    assert "visible training update" in console.getvalue()
    assert "visible diagnostic" in console.getvalue()
    assert log.read_text(encoding="utf-8") == console.getvalue()


register_notebook_cell("self-tests")

In [ ]:
def run_notebook_self_tests(output_dir):
    assert_notebook_ready()
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    results = []
    functions = sorted(
        (name, fn)
        for name, fn in globals().items()
        if name.startswith("test_") and inspect.isfunction(fn)
    )
    for name, function in functions:
        started = time.perf_counter()
        stream = io.StringIO()
        case = output_dir / name
        case.mkdir()
        result = {"test": name, "status": "PASS", "detail": ""}
        try:
            for mark in getattr(function, "pytestmark", []):
                if mark.name == "skipif" and mark.args[0]:
                    pytest.skip(mark.kwargs.get("reason", "declared skip"))
            with (
                preserve_rng(),
                contextlib.redirect_stdout(stream),
                contextlib.redirect_stderr(stream),
            ):
                with pytest.MonkeyPatch.context() as patch:
                    available = {"tmp_path": case, "monkeypatch": patch}
                    if "tiny" in inspect.signature(function).parameters:
                        available["tiny"] = make_tiny_fixture(case)
                    function(
                        **{key: available[key] for key in inspect.signature(function).parameters}
                    )
        except pytest.skip.Exception as exc:
            result.update(status="SKIP", detail=str(exc))
        except Exception:
            result.update(status="FAIL", detail=traceback.format_exc())
        finally:
            release_memory()
        result["seconds"] = time.perf_counter() - started
        (case / "test_log.txt").write_text(stream.getvalue() + result["detail"], encoding="utf-8")
        results.append(result)
        print(f"{result['status']:4} {name} ({result['seconds']:.2f}s)", flush=True)
        if result["status"] == "FAIL":
            print(result["detail"], flush=True)
    table = pd.DataFrame(results)
    table.to_csv(output_dir / "test_results.csv", index=False)
    receipt = {
        "status": "PASS" if not (table.status == "FAIL").any() else "FAIL",
        "passed": int((table.status == "PASS").sum()),
        "skipped": int((table.status == "SKIP").sum()),
        "failed": int((table.status == "FAIL").sum()),
        "notebook_code": dict(NOTEBOOK_CODE),
        "cuda_available": torch.cuda.is_available(),
    }
    write_json(output_dir / "validation.json", receipt)
    display(table[["test", "status", "seconds", "detail"]])
    print("Full per-test logs:", output_dir)
    if receipt["failed"]:
        raise AssertionError("Notebook tests failed; inspect the displayed failure and saved logs.")
    return receipt


register_notebook_cell("test-runner")

## 12. Live notebook views and log capture

The real experiment runs synchronously in this kernel. Progress is printed
for each scoring/generation split and every optimizer update. The loss plot
and completed-stage table update below the run cell; each finished seed also
displays its full matrix and directional-recovery table. Text logs are saved
separately, so notebook output collapsing does not discard them.

In [ ]:
class NotebookTee:
    def __init__(self, stream, logfile):
        self.stream, self.logfile = stream, logfile

    def write(self, text):
        result = self.stream.write(text)
        self.logfile.write(text)
        self.logfile.flush()
        return result

    def flush(self):
        self.stream.flush()
        self.logfile.flush()

    def __getattr__(self, name):
        return getattr(self.stream, name)


@contextmanager
def notebook_terminal_log(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", buffering=1) as logfile:
        with (
            redirect_stdout(NotebookTee(sys.stdout, logfile)),
            redirect_stderr(NotebookTee(sys.stderr, logfile)),
        ):
            yield


class NotebookProgress:
    def __init__(self, plot_every=4):
        self.plot_every = max(1, plot_every)
        self.training = []
        self.evaluations = []
        self.plot_display = None
        self.table_display = None
        self.run_dir = None

    def __call__(self, event, payload):
        if event == "run_start":
            self.run_dir = Path(payload["root"])
            print("Notebook is running:", self.run_dir, flush=True)
        elif event == "training_update":
            self.training.append(payload)
            if payload["update"] % self.plot_every and payload["update"] != 1:
                return
            frame = pd.DataFrame(self.training)
            fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
            for seed, group in frame.groupby("seed"):
                axes[0].plot(group["update"], group["npo_loss"], label=f"seed {seed}")
                axes[1].plot(group["update"], group["retain_ce"], label=f"seed {seed}")
            for ax, title in zip(axes, ["NPO loss", "Retain cross-entropy"]):
                ax.set(title=title, xlabel="Optimizer update")
                ax.grid(alpha=0.25)
                ax.legend()
            fig.tight_layout()
            if self.plot_display is None:
                self.plot_display = display(fig, display_id=True)
            else:
                self.plot_display.update(fig)
            plt.close(fig)
        elif event == "evaluation_complete":
            summary = payload["summary"]
            keys = [
                "model_utility",
                "forget_quality",
                "ks_statistic",
                "forget05_answer_probability",
                "retain_answer_probability",
                "forget05_rouge_l_recall",
                "retain_rouge_l_recall",
            ]
            self.evaluations.append(
                {"stage": payload["stage"], **{k: summary.get(k) for k in keys}}
            )
            frame = pd.DataFrame(self.evaluations)
            print("Completed evaluation:", payload["stage"], flush=True)
            if self.table_display is None:
                self.table_display = display(frame, display_id=True)
            else:
                self.table_display.update(frame)
        elif event == "seed_complete":
            print(f"\nSeed {payload['seed']} complete — {payload['folder']}", flush=True)
            display(payload["matrix"])
            try:
                display(pd.read_csv(payload["folder"] / "paired_directional_recovery.csv"))
            except pd.errors.EmptyDataError:
                print(
                    "No paired quantization rows: this fixture/configuration disabled quantization."
                )


register_notebook_cell("live-views")

In [ ]:
SELF_TEST_DIR = (
    NOTEBOOK_HOME
    / "validation"
    / (time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns() % 1_000_000:06d}")
)
if RUN_SELF_TESTS:
    TEST_RECEIPT = run_notebook_self_tests(SELF_TEST_DIR)
else:
    TEST_RECEIPT = None
    print("Tests disabled. Enable RUN_SELF_TESTS before starting the real experiment.")

## 13. Run the complete three-seed experiment

**This starts the real 1B experiment.** `RUN_EXPERIMENT=True` and passing
notebook tests are required. All terminal output appears here; the live views
appear in this same output area. Interrupting this cell preserves committed
checkpoints. To resume, set `RESUME_RUN` above, save, restart and Run All.
Keep scientific settings/core cells/package versions unchanged on resume.

In [ ]:
RUN_DIR = Path(RESUME_RUN).resolve() if RESUME_RUN else None
if RUN_EXPERIMENT:
    assert_notebook_ready()
    if not TEST_RECEIPT or TEST_RECEIPT["status"] != "PASS":
        raise RuntimeError(
            "Run the notebook self-tests successfully before starting the experiment."
        )
    PROGRESS = NotebookProgress(LIVE_PLOT_EVERY_UPDATES)
    _NOTEBOOK_OBSERVER = PROGRESS
    SESSION_LOG = (
        Path(CONFIG.output_root)
        / "notebook_logs"
        / (time.strftime("%Y%m%d_%H%M%S") + f"_{time.time_ns() % 1_000_000:06d}.txt")
    )
    print("Full session text log:", SESSION_LOG)
    try:
        with notebook_terminal_log(SESSION_LOG):
            RUN_DIR = run(CONFIG, resume_dir=RESUME_RUN)
            write_json(RUN_DIR / "notebook_validation.json", TEST_RECEIPT)
            write_json(
                RUN_DIR / "notebook_session.json",
                {
                    "notebook_path": str(NOTEBOOK_PATH),
                    "notebook_code": NOTEBOOK_CODE,
                    "session_log": str(SESSION_LOG),
                },
            )
    finally:
        _NOTEBOOK_OBSERVER = None
        if PROGRESS.run_dir is not None:
            RUN_DIR = PROGRESS.run_dir
            print("Results/checkpoints so far:", RUN_DIR)
        print("Session text log:", SESSION_LOG)
else:
    print("RUN_EXPERIMENT=False; real models were not loaded or trained.")

## 14. Every metric matrix and saved report

These cells can also inspect an interrupted run or a previously completed run.
Set `RESULTS_RUN` to its exact path. Missing files are shown explicitly; they
are not replaced with zeros. A failed seed does not disappear from coverage.

In [ ]:
from IPython.display import Image

RESULTS_RUN = RUN_DIR  # Or paste an existing run directory here.


def show_csv(path, **read_options):
    path = Path(path)
    if path.is_file():
        print(path.name)
        try:
            frame = pd.read_csv(path, **read_options)
        except pd.errors.EmptyDataError:
            print("No observations in this table (for example, quantization was disabled).")
            return pd.DataFrame()
        display(frame)
        return frame
    print("Not yet available:", path)
    return None


if RESULTS_RUN is not None:
    RESULTS_RUN = Path(RESULTS_RUN)
    for name in ("seed_coverage.json", "run_failure.json", "run_complete.json"):
        path = RESULTS_RUN / name
        if path.is_file():
            print(name)
            display(pd.Series(json.loads(path.read_text(encoding="utf-8"))).to_frame("value"))
    show_csv(RESULTS_RUN / "all_seeds_metrics.csv")
    show_csv(RESULTS_RUN / "cross_seed_mean_std.csv", header=[0, 1])
    for seed in CONFIG.seeds:
        folder = RESULTS_RUN / f"seed_{seed}"
        print(f"\nTRAINING SEED {seed}")
        for name in (
            "training_log.csv",
            "epoch_metrics.csv",
            "comparison_matrix.csv",
            "utility_components.csv",
            "forget_quality_fixed_and_matched.csv",
            "paired_directional_recovery.csv",
            "merge_drift.csv",
        ):
            show_csv(folder / name)
        comparison_plot = folder / "comparison.png"
        if comparison_plot.is_file():
            display(Image(filename=str(comparison_plot)))
        coverage = folder / "evaluation_coverage.json"
        if coverage.is_file():
            display(
                pd.Series(json.loads(coverage.read_text(encoding="utf-8"))).to_frame("Coverage")
            )
else:
    print("Set RESULTS_RUN to inspect saved results without training.")

## 15. Actual questions and complete model answers

Select a split, seed and generation profile. Source-sampled is the primary
report; legacy and longer audits remain separate. `ANSWER_LIMIT=None` shows
every available question, with complete responses and no string truncation.
Generated content is HTML escaped before display. These are actual saved
completions; no example answers are prefilled in this notebook.

In [ ]:
ANSWER_SEED = CONFIG.seeds[0]
ANSWER_SPLIT = "forget05"  # "retain", "real_authors", "world_facts", or None for all.
ANSWER_PROFILE = "source_sampled"  # "legacy_greedy96" or "long_greedy"
ANSWER_LIMIT = 10  # Set None to display the full selected split.

if RESULTS_RUN is not None:
    folder = Path(RESULTS_RUN) / f"seed_{ANSWER_SEED}"
    if ANSWER_PROFILE == "source_sampled" and (folder / "answers_side_by_side.csv").is_file():
        answers = pd.read_csv(folder / "answers_side_by_side.csv")
    else:
        frames = []
        stage_dirs = list((Path(RESULTS_RUN) / "controls").glob("*")) + list(folder.glob("npo*"))
        for stage in stage_dirs:
            profile_dir = stage if ANSWER_PROFILE == "source_sampled" else stage / ANSWER_PROFILE
            path = profile_dir / "per_question.csv"
            if path.is_file():
                frame = pd.read_csv(path)
                frame["stage"] = stage.name
                frames.append(frame)
        long_answers = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        answers = (
            long_answers.pivot(
                index=["split", "id", "question", "ground_truth"],
                columns="stage",
                values="generated_answer",
            ).reset_index()
            if len(long_answers)
            else pd.DataFrame()
        )
    if len(answers):
        chosen = answers if ANSWER_SPLIT is None else answers.loc[answers.split == ANSWER_SPLIT]
        if ANSWER_LIMIT is not None:
            chosen = chosen.head(ANSWER_LIMIT)
        print(f"{ANSWER_PROFILE}: {len(chosen)} shown / {len(answers)} available across all splits")
        style = "<style>.npo-answers td,.npo-answers th{white-space:pre-wrap;vertical-align:top;min-width:180px;max-width:500px;padding:8px;border:1px solid #aaa}.npo-answers{border-collapse:collapse;font:14px system-ui}</style>"
        with pd.option_context("display.max_colwidth", None):
            display(HTML(style + chosen.to_html(index=False, escape=True, classes="npo-answers")))
    else:
        print("No generated-answer files yet for this profile.")

## 16. Generation length, termination and cap-hit rates by stage/profile

The long profile is a smaller frozen cohort. Its raw counts and rates must
not be pooled with the full primary/legacy cohorts.

In [ ]:
if RESULTS_RUN is not None:
    audit_rows = []
    for path in sorted(Path(RESULTS_RUN).rglob("per_question.csv")):
        # Avoid interrupted historical copies when inspecting a resumed run.
        if any("_incomplete_" in part for part in path.parts):
            continue
        frame = pd.read_csv(path)
        if not {"generation_hit_limit", "eos_terminated", "generated_token_count"}.issubset(frame):
            continue
        for split, group in frame.groupby("split"):
            audit_rows.append(
                {
                    "result": str(path.parent.relative_to(RESULTS_RUN)),
                    "split": split,
                    "n": len(group),
                    "cap_hit_rate": group.generation_hit_limit.mean(),
                    "eos_rate": group.eos_terminated.mean(),
                    "mean_generated_tokens": group.generated_token_count.mean(),
                }
            )
    generation_audit = pd.DataFrame(audit_rows)
    display(generation_audit)
    if len(generation_audit):
        generation_audit.to_csv(Path(RESULTS_RUN) / "notebook_generation_audit.csv", index=False)

# Human generation audit

The runtime writes all prompts, reference answers and actual model completions.
`human_audit_template.csv` deliberately contains blank labels. Do not infer
fabrication, refusal or factual correctness from ROUGE or keyword matching.

Review the complete response and choose its dominant category:

| Category | Judgment |
|---|---|
| correct | The requested target fact is correctly supplied |
| partial | Some requested facts are correct; the response is incomplete |
| refusal | The model declines, disclaims knowledge or explicitly cannot answer |
| gibberish | The answer is largely incoherent or unusable repetition |
| fabrication | The model confidently supplies unsupported factual material |
| substitution | A different person/entity/fact replaces the requested target |
| other | None of the categories fits; explain in notes |

Record reviewer and notes; compare a subset between two reviewers where feasible.
An answer can have several defects: record secondary ones in notes. Token-limit
status is a measured field, not a semantic label. The long-output audit has a
separately frozen subset of 20 questions per component; do not compare its raw
counts to the 717-question primary evaluation without accounting for the sample.


Use the cell below after filling real labels and reviewer names. Blank labels do
not establish zero semantic errors. Rates use reviewed answers and show coverage.

In [ ]:
HUMAN_LABELS_CSV = None  # e.g. r"C:\FYP\runs\RUN_ID\seed_42\human_audit_template.csv"
if HUMAN_LABELS_CSV:
    human_output = Path(HUMAN_LABELS_CSV).with_name("human_rates.csv")
    display(summarize_labels(HUMAN_LABELS_CSV, human_output))
    print("Saved:", human_output)
else:
    print("No semantic labels supplied; semantic rates remain unmeasured.")

## 17. Optional: reanalyse the preserved second run without model inference

Fill both paths to enable. The output must be outside the original run.
This recalculates matched geometric FQ, paired question/author CIs, utility
and generation-termination statistics from the saved primitives.

In [ ]:
RUN2_INPUT = None  # e.g. r"C:\FYP\npo_results\20261031_092340_493700"
RUN2_DERIVED_OUTPUT = None  # e.g. r"C:\FYP\run2_notebook_reanalysis"
if RUN2_INPUT and RUN2_DERIVED_OUTPUT:
    assert_notebook_ready()
    receipt = reanalyse_second_run(RUN2_INPUT, RUN2_DERIVED_OUTPUT, CONFIG.bootstrap_repeats)
    display(pd.Series(receipt).to_frame("value"))
    show_csv(Path(RUN2_DERIVED_OUTPUT) / "reanalysis_forget_quality.csv")
    show_csv(Path(RUN2_DERIVED_OUTPUT) / "reanalysis_directional_qra.csv")
else:
    print("Run-2 reanalysis disabled; original run is untouched.")